# NB5: Fine-tune PREDATOR on NSS (GPU notebook)

**What this does**
- Starts from PREDATOR's pretrained indoor weights and **fine-tunes them on NSS scans of other buildings**: Bldg1, Bldg2 and Bldg6 for training, Bldg4 for validation.
- **Bldg3 and Bldg5 are never seen**, because they are our test buildings.
- Labels come from the NSS pair transforms:
  - same-stage pairs use the official transform, which is exact;
  - cross-stage pairs are first polished with a robust ICP, because the official cross-stage tilt is off by about 1°. Pairs that can't be verified are dropped.
- The input scale is the one NB4 chose on Bldg3 dev pairs (0.191), so training and testing use the same units.
- After every epoch: registration success on 30 held-out Bldg4 pairs. The best epoch is saved as `predator_nss_ft.pth`.

**Kaggle settings:** *Accelerator → GPU T4 x2*, *Internet → On*. Input: the NSS dataset (all buildings).

**Run order:**
1. `QUICK = True` first, about 20 min. Check that the loss goes down and nothing crashes.
2. Then `QUICK = False` → *Save & Run All (Commit)*. Training stops by itself before Kaggle's 12 h limit and keeps the best epoch.

**Then:** add this notebook's output as an input to NB4. NB4 finds `predator_nss_ft.pth` and runs the fine-tuned model (C-ft, D-ft) on the same 240 pairs.

## Setup
Kaggle settings: **Internet On** (for `pip install open3d`), **Accelerator GPU T4 x2** (PREDATOR needs the GPU; our methods are CPU). Add your NSS dataset (and `nss-prep` if you have it) with *Add Input*.

In [ ]:
import sys, subprocess
try:
    import open3d
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "open3d"], check=True)

The next cell writes the shared library (`nss_lib.py`, identical in every notebook). You don't need to read it to run the notebook; every algorithm is explained in the markdown cells and in `PROJECT_GUIDE.md`.

In [ ]:
%%writefile nss_lib.py
"""nss_lib.py -- shared library for the CV5570 NSS project.

Sections
  0. config + small geometry helpers
  1. data loading (format auto-detection), pose graph, pair generation
  2. cloud cache (voxel, normals, FPFH, structure masks)
  3. registration: baselines (RANSAC-6DoF, FGR), ours (BEV-XC, RANSAC-4DoF, hypothesis verification),
     one weighted point-to-plane solver for every refiner (ICP, Tukey, trimmed, GNC = method B v2)
  4. visibility: scanner origins, spherical depth maps, three-class change detection, injected objects
  5. building-scale evidence fusion
  6. plotting helpers
Every function is deterministic given C["SEED"].
"""
import os, re, json, time, math, zipfile, warnings
from pathlib import Path
from collections import OrderedDict, deque
import numpy as np
import pandas as pd
import open3d as o3d
from scipy.spatial import cKDTree
from scipy import ndimage

warnings.filterwarnings("ignore", category=RuntimeWarning)
o3d.utility.set_verbosity_level(o3d.utility.VerbosityLevel.Error)
REG = o3d.pipelines.registration

# =====================================================================================
# 0. CONFIG + GEOMETRY HELPERS
# =====================================================================================
C = dict(
    SEED=0, UP=2,
    VOXEL=0.05,            # m, working resolution
    TAU=0.15,              # m, change threshold (same as the earlier experiments)
    FOOT_CELL=0.25,        # m, XY cell for footprint overlap
    MAXD0=1.0,             # m, initial correspondence distance of every refiner (same for all = fair)
    DEPTH_RES="auto",      # deg, angular resolution of spherical depth maps ("auto" = from point density)
    DEPTH_MAX_PTS=1_500_000,
    FS_MARGIN_REL=0.02,    # free-space margin grows with range (beam footprint, grazing angles)
    ORIGIN_MODE="auto",    # "local" (scanner at fragment origin), "estimate", "auto"
    BEV_CELL=0.10, BEV_YAW_STEP=2.0, GNC_DECAY=1.15,
    SUCC_COARSE=(0.5, 5.0), SUCC_FINE=(0.10, 1.0),   # (m, deg): identical to the earlier runs
)
def configure(**kw):
    C.update(kw)

def tf(P, T): return P @ T[:3, :3].T + T[:3, 3]
def inv(T):
    Ti = np.eye(4); Ti[:3, :3] = T[:3, :3].T; Ti[:3, 3] = -T[:3, :3].T @ T[:3, 3]; return Ti
def rotz(deg):
    a = np.radians(deg); c, s = np.cos(a), np.sin(a)
    T = np.eye(4); T[:2, :2] = [[c, -s], [s, c]]; return T
def make_T(R=None, t=None):
    T = np.eye(4)
    if R is not None: T[:3, :3] = R
    if t is not None: T[:3, 3] = t
    return T
def hax():
    return [i for i in range(3) if i != C["UP"]]

def reg_errors(T_est, T_gt, src_pts):
    dR = T_est[:3, :3] @ T_gt[:3, :3].T
    rre = float(np.degrees(np.arccos(np.clip((np.trace(dR) - 1) / 2, -1, 1))))
    rte = float(np.linalg.norm(T_est[:3, 3] - T_gt[:3, 3]))
    disp = float(np.sqrt(np.mean(np.sum((tf(src_pts, T_est) - tf(src_pts, T_gt)) ** 2, 1))))
    ct, cr = C["SUCC_COARSE"]; ft, fr = C["SUCC_FINE"]
    return dict(rre=rre, rte=rte, disp=disp, succ_coarse=bool(rte < ct and rre < cr), succ_fine=bool(rte < ft and rre < fr))

def bench_frame(e, yaw_deg):
    """Realistic registration input: move the source to its own centroid and give it a random heading.
    Returns M (apply to the source). New GT = T_gt @ inv(M); an estimate T maps back via T @ M."""
    c = e["pts"].mean(0); Tc = make_T(t=-c)
    return rotz(yaw_deg) @ Tc if C["UP"] == 2 else _rot_up(yaw_deg) @ Tc

def failure_type(e):
    if e["succ_coarse"]: return "ok"
    if e["rre"] > 20: return "flip"        # wrong heading (symmetric layout)
    if e["rre"] < 5: return "slide"        # right heading, slid along a plane / repetitive structure
    return "other"

# =====================================================================================
# 1. DATA LOADING
# =====================================================================================
PC_EXT = {".ply", ".pcd", ".npz", ".bin"}
ANNOT_EXT = {".csv", ".json", ".txt", ".pkl", ".pickle", ".yaml", ".yml", ".log", ".npy"}
INDEX = {}

def find_and_extract(root, out):
    root, out = Path(root), Path(out); out.mkdir(parents=True, exist_ok=True)
    for z in sorted(root.rglob("*.zip")):
        dst = out / z.stem; marker = dst / ".done"
        if marker.exists(): continue
        print(f"extracting {z.name} ({z.stat().st_size/1e9:.2f} GB) -> {dst}")
        with zipfile.ZipFile(z) as zf: zf.extractall(dst)
        marker.touch()
    return [root, out]

def _is_pc(p):
    s = p.suffix.lower()
    if s in PC_EXT: return True
    if s == ".npy":
        try:
            a = np.load(p, mmap_mode="r"); return a.ndim == 2 and a.shape[1] >= 3 and a.shape[0] > 100
        except Exception: return False
    return False

def read_points(path):
    path = Path(path); s = path.suffix.lower()
    if s in (".ply", ".pcd"): return np.asarray(o3d.io.read_point_cloud(str(path)).points, dtype=np.float64)
    if s == ".npy": return np.load(path)[:, :3].astype(np.float64)
    if s == ".npz":
        z = np.load(path); a = z[[k for k in z.files if z[k].ndim == 2 and z[k].shape[1] >= 3][0]]
        return a[:, :3].astype(np.float64)
    if s == ".bin":
        a = np.fromfile(path, dtype=np.float32); return a.reshape(-1, 4 if a.size % 4 == 0 else 3)[:, :3].astype(np.float64)
    raise ValueError(path)

def _read_table(p):
    s = p.suffix.lower()
    try:
        if s == ".csv": return pd.read_csv(p)
        if s == ".json":
            obj = json.loads(p.read_text())
            if isinstance(obj, list): return pd.json_normalize(obj)
            if isinstance(obj, dict):
                vals = list(obj.values())
                if vals and all(isinstance(v, list) and v and isinstance(v[0], dict) for v in vals):
                    return pd.concat([pd.json_normalize(v).assign(_group=k) for k, v in obj.items()], ignore_index=True)
                if vals and all(isinstance(v, dict) for v in vals):
                    return pd.DataFrame.from_dict(obj, orient="index").reset_index().rename(columns={"index": "_key"})
                try: return pd.DataFrame(obj)
                except Exception: return None
        if s in (".pkl", ".pickle"):
            obj = pd.read_pickle(p)
            return obj if isinstance(obj, pd.DataFrame) else (pd.DataFrame(obj) if isinstance(obj, (list, dict)) else None)
        if s in (".txt", ".log"):
            df = pd.read_csv(p, sep=None, engine="python")
            return df if df.shape[1] > 1 else None
    except Exception:
        return None
    return None

NUM_RE = re.compile(r"[-+]?(?:\d+\.\d*|\.\d+|\d+)(?:[eE][-+]?\d+)?")
def to_mat(v):
    if v is None or (isinstance(v, float) and np.isnan(v)): return None
    try:
        arr = np.array([float(x) for x in NUM_RE.findall(v)]) if isinstance(v, str) else np.asarray(v, float).ravel()
    except Exception: return None
    if arr.size == 16: return arr.reshape(4, 4)
    if arr.size == 12: return np.vstack([arr.reshape(3, 4), [0, 0, 0, 1]])
    return None

def norm_key(v):
    s = str(v).strip().replace("\\", "/"); name = s.split("/")[-1]
    if name in INDEX: return name
    for ext in (".ply", ".pcd", ".npy", ".npz", ".bin"):
        if name.lower().endswith(ext) and name[: -len(ext)] in INDEX: return name[: -len(ext)]
    two = "/".join(s.split("/")[-2:])
    for ext in ("", ".ply", ".pcd"):
        k = two[: -len(ext)] if ext and two.endswith(ext) else two
        if k in INDEX: return k
    return Path(name).stem

def _match_rate(series):
    vals = series.dropna().astype(str).head(500)
    return 0.0 if len(vals) == 0 else float(np.mean([norm_key(v) in INDEX for v in vals]))

def _matrix_from_cols(df, prefix=None):
    groups = {}
    for c in df.columns:
        m = re.match(r"^(.*?)[_\-\[]?(\d)[_\-,\]]?(\d)?\]?$", str(c))
        if m and pd.api.types.is_numeric_dtype(df[c]): groups.setdefault(m.group(1), []).append(c)
    for pre, cols in groups.items():
        if prefix is not None and pre != prefix: continue
        if len(cols) in (12, 16):
            cols = sorted(cols, key=lambda c: [int(x) for x in re.findall(r"\d+", str(c))[-2:]])
            return pre, [to_mat(r) for r in df[cols].to_numpy(float)]
    return None, None

def _rt_cols(df):
    """Transform given as separate rotation (3x3 or quaternion x,y,z,w) and translation (3) columns."""
    rc = [c for c in df.columns if re.search(r"(^r$|rot)", str(c), re.I)]
    tc = [c for c in df.columns if re.search(r"(^t$|trans|^pos)", str(c), re.I) and c not in rc]
    for r_ in rc:
        for t_ in tc:
            def mk(rv, tv):
                try:
                    R = np.asarray(rv, float).ravel() if not isinstance(rv, str) else np.array([float(x) for x in NUM_RE.findall(rv)])
                    t = np.asarray(tv, float).ravel() if not isinstance(tv, str) else np.array([float(x) for x in NUM_RE.findall(tv)])
                except Exception: return None
                if t.size != 3: return None
                if R.size == 9: Rm = R.reshape(3, 3)
                elif R.size == 4:
                    from scipy.spatial.transform import Rotation
                    Rm = Rotation.from_quat(R).as_matrix()
                else: return None
                return make_T(Rm, t)
            mats = [mk(a, b) for a, b in zip(df[r_], df[t_])]
            if sum(m is not None for m in mats[:20]) >= min(5, len(mats)): return f"{r_}+{t_}", mats
    return None, None

def _matrix_col(df, explicit=None, prefix=None):
    if prefix: return _matrix_from_cols(df, prefix)
    cands = [explicit] if explicit else [c for c in df.columns if re.search(r"(^t$|trans|pose|rt|^gt|matrix|extrinsic|tsfm|xform|^tf$)", str(c), re.I)]
    cands = sorted(cands, key=lambda c: ("global" in str(c).lower(), len(str(c))))   # NSS: 'tsfm' before 'tsfm_global' 
    for c in cands:
        if c in df.columns:
            sample = df[c].dropna().head(5).map(to_mat)
            if len(sample) and sample.map(lambda m: m is not None).all(): return c, df[c].map(to_mat).tolist()
    pre, mats = _matrix_from_cols(df)
    if mats is not None: return pre, mats
    return _rt_cols(df)

SUFFIX_PAIRS = [("src", "tgt"), ("source", "target"), ("_a", "_b"), ("a", "b"), ("0", "1"), ("1", "2"), ("ref", "qry"),
                ("ref", "query"), ("i", "j"), ("anchor", "positive"), ("first", "second")]
def _find_pair_cols(df):
    best = None; cols = [str(c) for c in df.columns]
    for c1 in cols:
        for c2 in cols:
            if c1 == c2: continue
            for a, b in SUFFIX_PAIRS:
                if a in c1.lower() and b in c2.lower() and c1.lower().replace(a, "#", 1) == c2.lower().replace(b, "#", 1):
                    r = min(_match_rate(df[c1]), _match_rate(df[c2]))
                    if best is None or r > best[2]: best = (c1, c2, r)
    return best

def _parse_meta(key, path):
    s = f"{path.parent.parent.name}/{path.parent.name}/{key}"
    def g(pat):
        m = re.search(pat, s, re.I); return m.group(1) if m else None
    b = g(r"(b(?:ui)?ld(?:in)?g[_\-]?\d+)")
    st = g(r"stage[_\-]?(\d+)") or g(r"(?:epoch|time)[_\-]?(\d+)")
    sp = g(r"(?:spot|scan|frag(?:ment)?)[_\-]?(\d+)")
    return (b.capitalize() if b else "B?"), st, sp

def _mat_from_any(v):
    """A 4x4/3x4 matrix, or a dict holding one (any key) or rotation+translation entries."""
    M = to_mat(v) if not isinstance(v, dict) else None
    if M is not None: return M
    if isinstance(v, dict):
        rot = next((v[k] for k in v if re.search(r"(^r$|rot)", str(k), re.I)), None)
        tr = next((v[k] for k in v if re.search(r"(^t$|trans|^pos(ition)?$)", str(k), re.I)), None)
        if rot is not None and tr is not None:
            try:
                R = np.asarray(rot, float).ravel(); t = np.asarray(tr, float).ravel()
                if t.size == 3 and R.size == 9: return make_T(R.reshape(3, 3), t)
                if t.size == 3 and R.size == 4:
                    from scipy.spatial.transform import Rotation
                    return make_T(Rotation.from_quat(R).as_matrix(), t)
            except Exception: pass
        for k in v:
            if re.search(r"(pose|trans|matrix|extrinsic|^t$|gt)", str(k), re.I):
                M = _mat_from_any(v[k])
                if M is not None: return M
    return None

def _poses_in_json(obj, depth=0):
    """Walk any JSON structure; return {fragment: 4x4} for dicts keyed by fragment names or
    lists of records that carry a fragment name plus a pose."""
    out = {}
    if depth > 5: return out
    if isinstance(obj, dict):
        ks = list(obj.keys())[:500]
        hit = [k for k in ks if norm_key(k) in INDEX]
        if ks and len(hit) >= 0.5 * len(ks):
            for k in obj:
                if norm_key(k) in INDEX:
                    M = _mat_from_any(obj[k])
                    if M is not None: out[norm_key(k)] = M
            if out: return out
        for v in obj.values():
            if isinstance(v, (dict, list)): out.update(_poses_in_json(v, depth + 1))
    elif isinstance(obj, list) and obj and isinstance(obj[0], dict):
        name_keys = [k for k in obj[0] if isinstance(obj[0][k], str) and norm_key(obj[0][k]) in INDEX]
        if len(name_keys) == 1:                       # exactly one fragment name per record = per-fragment pose list
            nk = name_keys[0]
            for rec in obj:
                rest = {k: v for k, v in rec.items() if k != nk}
                M = _mat_from_any(rest)
                if M is not None and isinstance(rec.get(nk), str): out[norm_key(rec[nk])] = M
    return out

def load_nss(root, extract_dir, overrides=None, verbose=True):
    """Index point clouds, parse annotation tables. Returns dict D."""
    ov = dict(pair_file=None, src_col=None, tgt_col=None, T_col=None, T_prefix=None, overlap_col=None,
              frag_file=None, frag_key_col=None, pose_col=None, pose_prefix=None)
    ov.update(overrides or {})
    roots = find_and_extract(root, extract_dir)
    files = [p for r in roots for p in Path(r).rglob("*") if p.is_file() and not p.name.startswith(".")
             and "nss_out" not in p.parts and "nss_prep" not in p.parts]       # never re-read our own outputs
    pcs = [p for p in files if _is_pc(p)]
    INDEX.clear()
    for p in pcs:
        k = p.stem
        if k in INDEX and INDEX[k] != p: k = f"{p.parent.name}/{p.stem}"
        INDEX[k] = p
    pcset = set(pcs)
    annots = [p for p in files if p.suffix.lower() in ANNOT_EXT and p not in pcset]
    if verbose: print(f"{len(files)} files, {len(INDEX)} point clouds, {len(annots)} annotation-like files")
    if not INDEX:
        raise RuntimeError(
            f"No point clouds found under {root}. Attach the NSS dataset (pointclouds + annotation) with 'Add Input'. "
            "nss-prep alone is not enough: it only holds poses and pairs, the notebooks still read the scans.")
    tables = {}
    for p in sorted(annots)[:300]:
        df = _read_table(p)
        if df is not None and not df.empty: tables[str(p)] = df
    # per-fragment poses
    frag_pose, meta_cols = {}, {}
    for name, df in tables.items():
        if ov["frag_file"] and ov["frag_file"] not in name: continue
        kc = ov["frag_key_col"]
        if kc is None:
            rates = {c: _match_rate(df[c]) for c in df.columns if not pd.api.types.is_float_dtype(df[c])}
            if not rates: continue
            kc, r = max(rates.items(), key=lambda kv: kv[1])
            fp = _find_pair_cols(df)
            if r < 0.5 or (fp and fp[2] > 0.5): continue
        pc, mats = _matrix_col(df, ov["pose_col"], ov["pose_prefix"])
        if mats is None: continue
        for k, M in zip(df[kc], mats):
            if M is not None: frag_pose[norm_key(k)] = M
        for c in df.columns:
            if re.search(r"(bldg|building|stage|spot|scene|epoch)", str(c), re.I):
                meta_cols.setdefault(c, {}).update(dict(zip(df[kc].map(norm_key), df[c])))
        if verbose: print(f"[fragment poses] {Path(name).name}: key={kc} pose={pc} rows={len(df)}")
    # poses hidden inside JSON as {fragment_name: matrix} or {fragment_name: {rotation, translation}} (any depth)
    for p in annots:
        if p.suffix.lower() != ".json": continue
        try: obj = json.loads(p.read_text())
        except Exception: continue
        got = _poses_in_json(obj)
        new = {k: M for k, M in got.items() if k not in frag_pose}
        if new:
            frag_pose.update(new)
            if verbose: print(f"[fragment poses] {'/'.join(p.parts[-2:])}: {len(new)} poses found inside JSON")
    for p in annots:
        k = norm_key(p.name)
        if k in INDEX and k not in frag_pose and p.suffix.lower() in (".txt", ".npy"):
            try:
                M = to_mat(np.load(p) if p.suffix == ".npy" else p.read_text())
                if M is not None: frag_pose[k] = M
            except Exception: pass
    # pair tables
    rows = []
    for name, df in tables.items():
        if ov["pair_file"] and ov["pair_file"] not in name: continue
        if ov["src_col"]:
            if ov["src_col"] not in df.columns: continue
            sc, tc, r = ov["src_col"], ov["tgt_col"], 1.0
        else:
            fp = _find_pair_cols(df)
            if fp is None or fp[2] < 0.5: continue
            sc, tc, r = fp
        Tc, mats = _matrix_col(df, ov["T_col"], ov["T_prefix"])
        oc = ov["overlap_col"] or next((c for c in df.columns if "overlap" in str(c).lower()), None)
        if verbose: print(f"[pairs] {'/'.join(Path(name).parts[-2:])}: src={sc} tgt={tc} match={r:.0%} T={Tc} overlap={oc} n={len(df)}"
                          f"\n        columns: {list(df.columns)[:25]}")
        cols = list(df.columns)
        for i, row in enumerate(df.itertuples(index=False)):
            rd = dict(zip(cols, row))
            rows.append(dict(src=norm_key(rd[sc]), tgt=norm_key(rd[tc]), T_raw=(mats[i] if mats is not None else None),
                             overlap_annot=(float(rd[oc]) if oc is not None and pd.notna(rd[oc]) else np.nan),
                             ann_file=Path(name).name))
    pa = pd.DataFrame(rows, columns=["src", "tgt", "T_raw", "overlap_annot", "ann_file"])
    if len(pa):
        pa = pa[pa.src.isin(INDEX) & pa.tgt.isin(INDEX) & (pa.src != pa.tgt)]
        pa = pa.assign(uk=[tuple(sorted(x)) for x in zip(pa.src, pa.tgt)]).drop_duplicates("uk").drop(columns="uk").reset_index(drop=True)
    # fragment metadata
    fr = []
    for k, p in INDEX.items():
        b, st, sp = _parse_meta(k, p)
        for c, m in meta_cols.items():
            v = m.get(k); cl = str(c).lower()
            if v is None: continue
            if "bldg" in cl or "building" in cl or "scene" in cl: b = str(v)
            elif "stage" in cl or "epoch" in cl: st = str(v)
            elif "spot" in cl: sp = str(v)
        fr.append(dict(key=k, path=str(p), building=b, stage=st, spot=sp, size_mb=p.stat().st_size / 1e6, has_pose=k in frag_pose))
    frags = pd.DataFrame(fr).set_index("key")
    frags["stage_num"] = pd.to_numeric(frags.stage.astype(str).str.extract(r"(\d+)")[0], errors="coerce")
    if verbose:
        print(f"annotated pairs: {len(pa)} | fragment poses: {len(frag_pose)}")
        print(frags.groupby("building").agg(fragments=("path", "size"), stages=("stage_num", "nunique"),
                                            size_gb=("size_mb", lambda x: x.sum() / 1e3)).round(2).to_string())
    return dict(index=dict(INDEX), frags=frags, pairs_annot=pa, frag_pose=frag_pose, tables=tables)

# =====================================================================================
# 2. CLOUD CACHE
# =====================================================================================
class CloudCache:
    """Voxelised fragments with normals (oriented to the scanner), structure masks and lazy FPFH."""
    def __init__(self, max_items=400):
        self.d = OrderedDict(); self.max_items = max_items; self.origins = {}
    def origin(self, key):
        return self.origins.get(key, np.zeros(3))
    def __call__(self, key):
        if key in self.d:
            self.d.move_to_end(key); return self.d[key]
        P = read_points(INDEX[key])
        e = make_entry(P, self.origin(key))
        self.d[key] = e
        if len(self.d) > self.max_items: self.d.popitem(last=False)
        return e
    def fpfh(self, key): return entry_fpfh(self(key))

def make_entry(P, origin=None, voxel=None):
    v = voxel or C["VOXEL"]
    pcd = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(P)).voxel_down_sample(v)
    pcd.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=3 * v, max_nn=30))
    if origin is not None: pcd.orient_normals_towards_camera_location(np.asarray(origin, float))
    return finish_entry(pcd)

def finish_entry(pcd):
    pts = np.asarray(pcd.points).copy(); nrm = np.asarray(pcd.normals).copy()
    nu = np.abs(nrm[:, C["UP"]])
    return dict(pcd=pcd, pts=pts, nrm=nrm, tree=cKDTree(pts), wall=nu < 0.3, horiz=nu > 0.9)

def entry_fpfh(e):
    if "fpfh" not in e:
        f = REG.compute_fpfh_feature(e["pcd"], o3d.geometry.KDTreeSearchParamHybrid(radius=5 * C["VOXEL"], max_nn=100))
        e["fpfh"] = f; e["fpfh_np"] = np.asarray(f.data).T.copy()
    return e["fpfh"]

def transformed_entry(e, T):
    """Rigidly moved copy of an entry. FPFH is rotation invariant, so the features are reused (no recompute)."""
    pts = tf(e["pts"], T); nrm = e["nrm"] @ T[:3, :3].T
    pcd = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(pts)); pcd.normals = o3d.utility.Vector3dVector(nrm)
    out = finish_entry(pcd)
    if "fpfh" not in e: entry_fpfh(e)
    out["fpfh"] = e["fpfh"]; out["fpfh_np"] = e["fpfh_np"]
    return out

# ---------- ground truth: convention check + pose graph ----------
def _fitness(src_pts, tgt, T, thr):
    d, _ = tgt["tree"].query(tf(src_pts, T), distance_upper_bound=thr * 10)
    return float(np.mean(d < thr))

def check_convention(D, building, cloud, n=30, verbose=True):
    """Try every GT convention on a few pairs; scored on walls only (floors make any height-correct guess look good)."""
    pa = D["pairs_annot"]; frags = D["frags"]; fp = D["frag_pose"]
    pa = pa[pa.src.map(frags.building) == building]
    if pa.empty:     # no pair file: poses only
        ks = list(frags[(frags.building == building) & frags.has_pose].index)
        pa = pd.DataFrame([dict(src=a, tgt=b, T_raw=None) for a, b in zip(ks[:-1], ks[1:])])
    if "overlap_annot" in pa and pa.overlap_annot.notna().any(): pa = pa.sort_values("overlap_annot", ascending=False)
    res = []
    for row in pa.head(n).itertuples():
        S, Tg = cloud(row.src), cloud(row.tgt)
        cands = {"identity": np.eye(4)}
        if getattr(row, "T_raw", None) is not None: cands.update(pair_T=row.T_raw, pair_T_inv=inv(row.T_raw))
        if row.src in fp and row.tgt in fp:
            cands.update(pose_frag2world=inv(fp[row.tgt]) @ fp[row.src], pose_world2frag=fp[row.tgt] @ inv(fp[row.src]))
        for name, T in cands.items():
            res.append(dict(src=row.src, tgt=row.tgt, cand=name, fitness=_fitness(S["pts"][S["wall"]], Tg, T, 1.5 * C["VOXEL"])))
    res = pd.DataFrame(res)
    votes = res.loc[res.groupby(["src", "tgt"]).fitness.idxmax(), "cand"].value_counts()
    med = res.groupby("cand").fitness.median().sort_values(ascending=False)
    conv = med.index[0]; fit = float(med.iloc[0])      # best median fit over pairs (votes tie when scanners barely moved)
    if verbose: print("median wall fitness per candidate:", med.round(3).to_dict())
    if verbose:
        print(res.pivot_table(index=["src", "tgt"], columns="cand", values="fitness").round(3).to_string())
        print(f"-> convention {conv!r} (votes {votes.to_dict()}), median wall fitness {fit:.2f}")
        if fit < 0.3: print("!! low fitness: check units / pose columns before trusting anything")
        if set(res.cand) == {"identity"}:
            print("   (no transforms in the annotations: only 'identity' could be tested. Several pairs scoring >0.5 means the\n"
                  "    fragments are stored in one common frame; low scores are low-overlap or heavily changed pairs.)")
    return conv, fit, res

def world_poses(D, building, conv, cloud=None, verbose=True):
    """Pose of every fragment in one frame. Per-fragment poses if available, else chained through the pair graph
    (this is what lets us create MANY cross-stage pairs even if only a few are annotated).
    Robust chaining: some annotated transforms are wrong, and one bad edge in a naive chain corrupts everything after
    it. So every annotated edge is scored by how well it aligns the two fragments' WALLS, and the poses are chained
    along the MAXIMUM SPANNING TREE of that score. All other edges then serve as independent loop-closure checks."""
    frags = D["frags"]; fp = D["frag_pose"]
    keys = list(frags[frags.building == building].index)
    if conv == "identity":
        return ({k: np.eye(4) for k in keys}, {k: 0 for k in keys},
                dict(method="fragments already in a common frame (GT = identity)", n_posed=len(keys), n_total=len(keys),
                     components=1, largest_component=len(keys), loop_residual_med_m=None, loop_residual_max_m=None))
    if conv.startswith("pose_"):
        W = {k: (fp[k] if conv == "pose_frag2world" else inv(fp[k])) for k in keys if k in fp}
        return W, {k: 0 for k in W}, dict(method="per-fragment poses", n_posed=len(W), n_total=len(keys))
    pa = D["pairs_annot"]; pa = pa[pa.src.isin(keys) & pa.tgt.isin(keys) & pa.T_raw.notna()].reset_index(drop=True)
    Ts = [r.T_raw if conv == "pair_T" else inv(r.T_raw) for r in pa.itertuples()]          # src -> tgt
    if cloud is not None:
        t0 = time.time(); w = []
        for i, (r, T) in enumerate(zip(pa.itertuples(), Ts)):
            S_, T_ = cloud(r.src), cloud(r.tgt)
            w.append(_fitness(S_["pts"][S_["wall"]][::3], T_, T, 1.5 * C["VOXEL"]))
            if verbose and i % 300 == 0: print(f"  scoring annotated edges {i}/{len(pa)} ({time.time()-t0:.0f}s)")
        w = np.array(w)
    else:
        w = np.ones(len(pa))
    pa["edge_fit"] = w
    # Kruskal: strongest edges first
    parent = {k: k for k in keys}
    def find(x):
        while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]
        return x
    adj = {}
    for i in np.argsort(-w):
        r = pa.iloc[i]; a_, b_ = find(r.src), find(r.tgt)
        if a_ == b_: continue
        parent[a_] = b_; T = Ts[i]
        adj.setdefault(r.src, []).append((r.tgt, inv(T)))       # W[tgt] = W[src] @ inv(T)
        adj.setdefault(r.tgt, []).append((r.src, T))            # W[src] = W[tgt] @ T
    W, comp = {}, {}; ci = 0
    for start in sorted(adj, key=lambda k: -len(adj[k])):
        if start in W: continue
        W[start] = np.eye(4); comp[start] = ci; q = deque([start])
        while q:
            a_ = q.popleft()
            for b_, M in adj[a_]:
                if b_ not in W:
                    W[b_] = W[a_] @ M; comp[b_] = ci; q.append(b_)
        ci += 1
    # loop-closure residual of EVERY annotated edge vs the chained poses = how many annotations agree with the tree
    res = []
    for r, T in zip(pa.itertuples(), Ts):
        if comp.get(r.src) is not None and comp.get(r.src) == comp.get(r.tgt):
            E = inv(W[r.tgt]) @ W[r.src] @ inv(T); res.append(np.linalg.norm(E[:3, 3]))
        else: res.append(np.nan)
    pa["loop_residual_m"] = res
    D["edges_scored"] = pa
    sizes = pd.Series(comp).value_counts()
    rr = pa.loop_residual_m.dropna()
    info = dict(method="robust pose graph (max spanning tree of wall-fit) from annotated pair transforms",
                n_posed=len(W), n_total=len(keys), components=int(ci), largest_component=int(sizes.iloc[0]) if len(sizes) else 0,
                edges=len(pa), edge_fit_median=float(np.median(w)) if len(w) else None,
                loop_residual_med_m=float(rr.median()) if len(rr) else None,
                share_edges_consistent_10cm=float((rr < 0.10).mean()) if len(rr) else None)
    return W, comp, info

def footprint(P, cell=None):
    cell = cell or C["FOOT_CELL"]; ax = hax()
    return set(map(tuple, np.floor(P[:, ax] / cell).astype(np.int64)))

def make_pairs(D, building, W, comp, cloud, min_ov=0.3, min_geom=0.10, seed=0, verbose=True):
    """All fragment pairs (same component) with XY-footprint overlap >= min_ov and at least min_geom of their
    wall points overlapping under GT (otherwise no method could register them), labelled same/cross stage."""
    frags = D["frags"]; keys = [k for k in frags[frags.building == building].index if k in W]
    t0 = time.time(); feet = {}
    for i, k in enumerate(keys):
        feet[k] = footprint(tf(cloud(k)["pts"], W[k]))
        if verbose and i % 25 == 0: print(f"  footprints {i}/{len(keys)} ({time.time()-t0:.0f}s)")
    ann = set(map(tuple, map(sorted, zip(D["pairs_annot"].src, D["pairs_annot"].tgt))))
    rows = []
    for i, a in enumerate(keys):
        for b in keys[i + 1:]:
            if comp.get(a) != comp.get(b): continue
            fa, fb = feet[a], feet[b]
            ov = len(fa & fb) / max(1, min(len(fa), len(fb)))
            if ov < min_ov: continue
            sa, sb = frags.stage_num[a], frags.stage_num[b]
            if (sa if pd.notna(sa) else 0) > (sb if pd.notna(sb) else 0): a2, b2 = b, a
            else: a2, b2 = a, b
            Tab = inv(W[b2]) @ W[a2]
            # geometric overlap on vertical structure (what registration can actually use)
            S_, T_ = cloud(a2), cloud(b2)
            g1 = _fitness(S_["pts"][S_["wall"]][::4], T_, Tab, 2 * C["VOXEL"])
            g2 = _fitness(T_["pts"][T_["wall"]][::4], S_, inv(Tab), 2 * C["VOXEL"])
            rows.append(dict(src=a2, tgt=b2, stage_src=frags.stage_num[a2], stage_tgt=frags.stage_num[b2],
                             cross_stage=bool(sa != sb), ov_foot=ov, ov_geom=min(g1, g2),
                             annotated=tuple(sorted((a, b))) in ann, T_gt=Tab))
    P = pd.DataFrame(rows)
    pa = D["pairs_annot"]
    if len(P) and pa.overlap_annot.notna().any():
        ao = {tuple(sorted(k)): v for k, v in zip(zip(pa.src, pa.tgt), pa.overlap_annot)}
        P["overlap_annot"] = [ao.get(tuple(sorted((a, b))), np.nan) for a, b in zip(P.src, P.tgt)]
        m = P.overlap_annot.notna()
        if m.sum() >= 10:
            rho = P.loc[m, ["overlap_annot", "ov_geom"]].corr("spearman").iloc[0, 1]
            if verbose:
                print(f"  GT sanity check: Spearman(annotated overlap, our wall overlap under GT) = {rho:.2f} on {int(m.sum())} pairs")
                share = (D.get("edges_scored") is not None and len(D["edges_scored"]) and
                         float((D["edges_scored"].loop_residual_m < 0.10).mean())) or None
                if rho < 0.4 and (share is None or share < 0.6):
                    print("  !! LOW: the ground truth used here is probably WRONG (fragments misaligned). Do not run NB1-NB3; send the NB0 output.")
                elif rho < 0.4:
                    print("     (weak check: NSS defines overlap differently from our wall overlap; the pose graph agrees with "
                          f"{share:.0%} of the annotated transforms, which is the stronger evidence)")
    n0 = len(P); P = P[P.ov_geom >= min_geom].reset_index(drop=True)
    if verbose: print(f"  dropped {n0 - len(P)} pairs with wall overlap < {min_geom}")
    rng = np.random.default_rng(seed)
    P["pair_id"] = P.src + "__" + P.tgt
    P["yaw_deg"] = rng.uniform(0, 360, len(P))          # random heading applied to the source = fair benchmark
    P["ov_bin"] = pd.cut(P.ov_geom, [0, .2, .35, .5, 1.01], labels=["<0.2", "0.2-0.35", "0.35-0.5", ">0.5"])
    # location split: 8 m cells of the pair midpoint, ~30% dev
    cen = {k: tf(cloud(k)["pts"].mean(0)[None], W[k])[0] for k in set(P.src) | set(P.tgt)} if len(P) else {}
    mids = np.array([(cen[r.src] + cen[r.tgt]) / 2 for r in P.itertuples()]) if len(P) else np.zeros((0, 3))
    ax = hax()
    P["loc"] = [f"{int(m[ax[0]] // 8)}_{int(m[ax[1]] // 8)}" for m in mids]
    locs = np.array(sorted(P["loc"].unique())); rng.shuffle(locs)
    dev = set(locs[: max(1, int(round(0.3 * len(locs))))]) if len(locs) > 1 else set()
    if len(locs) <= 1: print("!! only one location cell: everything goes to eval")
    P["split"] = np.where(P["loc"].isin(dev), "dev", "eval")
    if verbose:
        print(f"pairs with footprint overlap >= {min_ov}: {len(P)}")
        print(P.groupby(["split", "cross_stage"]).size().unstack(fill_value=0).to_string())
    return P

def stratified(df, n, by=("cross_stage", "ov_bin"), seed=0):
    if len(df) == 0: return df
    ng = max(1, df.groupby(list(by), observed=True).ngroups)
    return df.sample(frac=1, random_state=seed).groupby(list(by), observed=True).head(max(1, int(np.ceil(n / ng)))).reset_index(drop=True)

# =====================================================================================
# 3. REGISTRATION
# =====================================================================================
# ---------- baselines ----------
def reg_ransac6(S, Tg, iters=100_000):
    d = 1.5 * C["VOXEL"]
    r = REG.registration_ransac_based_on_feature_matching(
        S["pcd"], Tg["pcd"], entry_fpfh(S), entry_fpfh(Tg), True, d, REG.TransformationEstimationPointToPoint(False), 3,
        [REG.CorrespondenceCheckerBasedOnEdgeLength(0.9), REG.CorrespondenceCheckerBasedOnDistance(d)],
        REG.RANSACConvergenceCriteria(iters, 0.999))
    return np.asarray(r.transformation).copy()

def reg_fgr(S, Tg):
    r = REG.registration_fgr_based_on_feature_matching(S["pcd"], Tg["pcd"], entry_fpfh(S), entry_fpfh(Tg),
        REG.FastGlobalRegistrationOption(maximum_correspondence_distance=1.5 * C["VOXEL"]))
    return np.asarray(r.transformation).copy()

# ---------- hypothesis verification on STRUCTURE (walls/columns), not floors ----------
def structure_score(S, Tg, T, n=6000, seed=0):
    """Fraction of the source's vertical-structure points that land on target geometry.
    Floors/ceilings are excluded: they match under any horizontal slide, which is exactly the RANSAC failure mode."""
    idx = np.flatnonzero(S["wall"])
    if len(idx) < 50: idx = np.arange(len(S["pts"]))
    if len(idx) > n: idx = np.random.default_rng(seed).choice(idx, n, replace=False)
    d, j = Tg["tree"].query(tf(S["pts"][idx], T), distance_upper_bound=4 * C["VOXEL"])
    ok = np.isfinite(d)
    if not ok.any(): return 0.0
    nr = np.abs(np.einsum("ij,ij->i", tf(S["pts"][idx[ok]], T) - Tg["pts"][j[ok]], Tg["nrm"][j[ok]]))
    return float(np.sum(nr < 1.5 * C["VOXEL"]) / len(idx))

def feature_corr(S, Tg, seed=0):
    """Mutual nearest neighbours in FPFH space on non-horizontal points (cached per source/target pair)."""
    key = id(Tg)
    if S.get("_corr_key") == key: return S["_corr"]
    rng = np.random.default_rng(seed)
    entry_fpfh(S); entry_fpfh(Tg)
    ms = np.flatnonzero(~S["horiz"]); mt = np.flatnonzero(~Tg["horiz"])
    if len(ms) < 30 or len(mt) < 30: ms, mt = np.arange(len(S["pts"])), np.arange(len(Tg["pts"]))
    fs, ft = S["fpfh_np"][ms], Tg["fpfh_np"][mt]
    _, i_st = cKDTree(ft).query(fs, k=1, workers=-1); _, i_ts = cKDTree(fs).query(ft, k=1, workers=-1)
    mutual = i_ts[i_st] == np.arange(len(ms))
    cs, ct = ms[mutual], mt[i_st[mutual]]
    if len(cs) < 10: cs, ct = ms, mt[i_st]          # fall back to one-way matches
    if len(cs) > 6000:
        k = rng.choice(len(cs), 6000, replace=False); cs, ct = cs[k], ct[k]
    S["_corr_key"] = key; S["_corr"] = (S["pts"][cs], Tg["pts"][ct])
    return S["_corr"]

def feature_score(S, Tg, T):
    """Share of mutual FPFH matches that agree with T (within 3 voxels). Independent of the wall-overlap cue:
    a symmetric look-alike alignment can overlap walls well, but the local-shape matches will not agree with it."""
    P, Q = feature_corr(S, Tg)
    return float(np.mean(np.linalg.norm(tf(P, T) - Q, axis=1) < 3 * C["VOXEL"])) if len(P) else 0.0

# ---------- ours 1: gravity-aware RANSAC (4-DoF, 2-point minimal sample) ----------
def _prior_ok(cx, cy, prior):
    """Hypotheses that put the source centroid within prior radius (horizontal) of the rough position."""
    return np.hypot(cx - prior[0][0], cy - prior[0][1]) <= prior[1]

def reg_ransac4(S, Tg, n_hyp=30000, topk=8, seed=0, return_hyps=False, prior=None):
    """Scanners on site are levelled, so only yaw + 3D translation are unknown.
    A 2-correspondence minimal sample fixes (yaw, t): fewer random draws, no tilted solutions, no 3-point degeneracies.
    prior=(xy, radius): optional rough position of the source centroid in the target frame; hypotheses outside are
    discarded before scoring (operators know roughly where the scanner stood)."""
    rng = np.random.default_rng(seed); v = C["VOXEL"]
    P, Q = feature_corr(S, Tg, seed)
    ax0, ax1 = hax(); up = C["UP"]
    i = rng.integers(0, len(P), n_hyp); j = rng.integers(0, len(P), n_hyp)
    dp, dq = P[j] - P[i], Q[j] - Q[i]
    lp, lq = np.hypot(dp[:, ax0], dp[:, ax1]), np.hypot(dq[:, ax0], dq[:, ax1])
    ok = (lp > 0.5) & (np.abs(lp - lq) < 3 * v) & (np.abs(dp[:, up] - dq[:, up]) < 3 * v)
    i, j, dp, dq = i[ok], j[ok], dp[ok], dq[ok]
    if len(i) == 0: return (np.eye(4), []) if return_hyps else np.eye(4)
    yaw = np.arctan2(dq[:, ax1], dq[:, ax0]) - np.arctan2(dp[:, ax1], dp[:, ax0])
    c, s = np.cos(yaw), np.sin(yaw)
    def rot(X):
        Y = X.copy(); Y[:, ax0] = c * X[:, ax0] - s * X[:, ax1]; Y[:, ax1] = s * X[:, ax0] + c * X[:, ax1]; return Y
    t = 0.5 * ((Q[i] - rot(P[i])) + (Q[j] - rot(P[j])))
    if prior is not None:
        cs = S["pts"].mean(0)
        keep = _prior_ok(c * cs[ax0] - s * cs[ax1] + t[:, ax0], s * cs[ax0] + c * cs[ax1] + t[:, ax1], prior)
        if not keep.any(): return (np.eye(4), []) if return_hyps else np.eye(4)
        yaw, c, s, t = yaw[keep], c[keep], s[keep], t[keep]
    # score every hypothesis by correspondence inliers (chunked, vectorised)
    sel = np.arange(len(P)) if len(P) <= 1500 else rng.choice(len(P), 1500, replace=False)
    Psub, Qsub = P[sel], Q[sel]
    score = np.zeros(len(yaw)); thr2 = (3 * v) ** 2
    for a in range(0, len(yaw), 2000):
        cc, ss, tt = c[a:a+2000, None], s[a:a+2000, None], t[a:a+2000]
        X0 = cc * Psub[None, :, ax0] - ss * Psub[None, :, ax1] + tt[:, None, ax0] - Qsub[None, :, ax0]
        X1 = ss * Psub[None, :, ax0] + cc * Psub[None, :, ax1] + tt[:, None, ax1] - Qsub[None, :, ax1]
        X2 = Psub[None, :, up] + tt[:, None, up] - Qsub[None, :, up]
        score[a:a+2000] = ((X0**2 + X1**2 + X2**2) < thr2).sum(1)
    order = np.argsort(-score); hyps = []
    for o in order:
        Th = np.eye(4); R2 = np.array([[c[o], -s[o]], [s[o], c[o]]])
        Th[np.ix_([ax0, ax1], [ax0, ax1])] = R2; Th[:3, 3] = t[o]
        if all(np.linalg.norm(Th[:3, 3] - h[:3, 3]) > 0.5 or abs(_yaw_diff(Th, h)) > 5 for h in hyps): hyps.append(Th)
        if len(hyps) >= topk: break
    best = max(hyps, key=lambda T: structure_score(S, Tg, T)) if hyps else np.eye(4)
    return (best, hyps) if return_hyps else best

def _yaw_of(T):
    a0, a1 = hax(); return np.degrees(np.arctan2(T[a1, a0], T[a0, a0]))
def _yaw_diff(A, B):
    return (( _yaw_of(A) - _yaw_of(B) + 180) % 360) - 180

# ---------- ours 2: BEV cross-correlation (correspondence-free, globally exhaustive) ----------
def _raster(XY, origin, shape, cell, blur=1.0):
    ij = np.floor((XY - origin) / cell).astype(np.int64)
    ok = (ij[:, 0] >= 0) & (ij[:, 1] >= 0) & (ij[:, 0] < shape[0]) & (ij[:, 1] < shape[1])
    img = np.zeros(shape, np.float32); img[ij[ok, 0], ij[ok, 1]] = 1.0      # binary: density-independent, change-robust
    return ndimage.gaussian_filter(img, blur) if blur else img

def _z_offset(S, Tg, bin_=0.02):
    zs = S["pts"][S["horiz"], C["UP"]]; zt = Tg["pts"][Tg["horiz"], C["UP"]]
    if len(zs) < 50 or len(zt) < 50: return float(np.median(Tg["pts"][:, C["UP"]]) - np.median(S["pts"][:, C["UP"]]))
    lo = min(zs.min(), zt.min()) - 1; hi = max(zs.max(), zt.max()) + 1
    b = np.arange(lo, hi + bin_, bin_)
    hs = np.histogram(zs, b)[0].astype(float); ht = np.histogram(zt, b)[0].astype(float)
    hs, ht = np.sqrt(hs), np.sqrt(ht)             # compress: floor slabs should not drown ceilings
    n = len(b) - 1; F = np.fft.rfft(ht, 2 * n) * np.conj(np.fft.rfft(hs, 2 * n))
    cc = np.fft.irfft(F, 2 * n); k = int(np.argmax(cc)); k = k - 2 * n if k > n else k
    return float(k * bin_)

def reg_bev(S, Tg, cell=None, yaw_step=None, topk=6, return_debug=False, prior=None):
    """Gravity-aligned bird's-eye-view cross-correlation.
    Walls/columns -> binary occupancy image. For every yaw (sweep), one FFT cross-correlation finds the best
    translation over the WHOLE plane at once: no features, no random sampling, globally optimal on the grid.
    Coarse-to-fine: yaw sweep on a 2.5x coarser grid, then refine the best yaws on the fine grid."""
    from scipy.fft import rfft2, irfft2, next_fast_len
    cell = cell or C["BEV_CELL"]; yaw_step = yaw_step or C["BEV_YAW_STEP"]; ax = hax()
    rng = np.random.default_rng(0)
    ws = S["pts"][S["wall"]][:, ax]; wt = Tg["pts"][Tg["wall"]][:, ax]
    if len(ws) < 50 or len(wt) < 50: ws, wt = S["pts"][:, ax], Tg["pts"][:, ax]
    if len(ws) > 60000: ws = ws[rng.choice(len(ws), 60000, replace=False)]
    if len(wt) > 200000: wt = wt[rng.choice(len(wt), 200000, replace=False)]
    rad = np.linalg.norm(ws, axis=1); keep = rad <= np.percentile(rad, 98); ws = ws[keep]
    rs = rad[keep].max() + 0.5
    tz = _z_offset(S, Tg)
    cs_xy = S["pts"].mean(0)[ax]
    levels = {}
    def level(c):
        if c not in levels:
            lo = wt.min(0) - rs; shape = tuple(int(x) for x in np.ceil((wt.max(0) + rs - lo) / c) + 1)
            ss = int(np.ceil(2 * rs / c)) + 1
            fsh = tuple(next_fast_len(n + ss) for n in shape)
            It = _raster(wt, lo, shape, c)
            levels[c] = dict(lo=lo, ss=ss, fsh=fsh, F=rfft2(It, fsh), n=np.sqrt((It ** 2).sum()), It=It)
        return levels[c]
    def corr_at(yaw, c, full=False):
        L_ = level(c); a = np.radians(yaw); R = np.array([[np.cos(a), -np.sin(a)], [np.sin(a), np.cos(a)]])
        so = -np.array([rs, rs]); Is = _raster(ws @ R.T, so, (L_["ss"], L_["ss"]), c)
        cc = irfft2(L_["F"] * np.conj(rfft2(Is, L_["fsh"])), L_["fsh"], workers=-1)
        if prior is not None:
            # translation t = lo + k*c - so; the source centroid lands at R*cs + t; keep only shifts near the prior
            ku = [np.where(np.arange(f) < f // 2, np.arange(f), np.arange(f) - f) for f in L_["fsh"]]
            tx = L_["lo"][0] + ku[0] * c - so[0] + (R @ cs_xy)[0] - prior[0][0]
            ty = L_["lo"][1] + ku[1] * c - so[1] + (R @ cs_xy)[1] - prior[0][1]
            mask = (tx[:, None] ** 2 + ty[None, :] ** 2) <= prior[1] ** 2
            if mask.any(): cc = np.where(mask, cc, -np.inf)
        k = np.unravel_index(int(np.argmax(cc)), cc.shape)
        peak = cc[k] / (L_["n"] * np.sqrt((Is ** 2).sum()) + 1e-9)
        sub = []
        for d in range(2):
            km = list(k); kp = list(k); km[d] -= 1; kp[d] = (kp[d] + 1) % cc.shape[d]
            y0, y1, y2 = cc[tuple(km)], cc[k], cc[tuple(kp)]; den = y0 - 2 * y1 + y2
            ok_ = np.isfinite(y0) and np.isfinite(y2) and abs(den) > 1e-12
            sub.append(0.5 * (y0 - y2) / den if ok_ else 0.0)
        kk = np.array([(k[d] if k[d] < L_["fsh"][d] // 2 else k[d] - L_["fsh"][d]) + sub[d] for d in range(2)])
        txy = L_["lo"] + kk * c - so
        return (peak, txy, cc) if full else (peak, txy)
    coarse = 2.5 * cell
    yaws = np.arange(0, 360, yaw_step); sc = np.array([corr_at(y, coarse)[0] for y in yaws])
    loc = [i for i in range(len(yaws)) if sc[i] >= sc[i - 1] and sc[i] >= sc[(i + 1) % len(yaws)]]
    loc = sorted(loc, key=lambda i: -sc[i])[:topk]
    hyps = []
    for i in loc:
        cand = []
        for y in np.arange(yaws[i] - yaw_step, yaws[i] + yaw_step + 1e-9, 0.5):
            p, t = corr_at(y, cell); cand.append((p, y, t))
        p0, y0, _ = max(cand, key=lambda r: r[0])
        for y in (y0 - 0.25, y0 + 0.25):
            p, t = corr_at(y, cell); cand.append((p, y, t))
        p, y, t = max(cand, key=lambda r: r[0])
        Th = rotz(y) if C["UP"] == 2 else _rot_up(y)
        Th[ax[0], 3], Th[ax[1], 3] = t; Th[C["UP"], 3] = tz
        hyps.append((Th, float(p)))
    Tbest = max(hyps, key=lambda h: structure_score(S, Tg, h[0]))[0]
    if return_debug:
        _, _, cc = corr_at(_yaw_of(Tbest), cell, full=True)
        return Tbest, dict(yaws=yaws, scores=sc, cc=np.fft.fftshift(np.where(np.isfinite(cc), cc, 0)), It=level(cell)["It"], hyps=hyps)
    return Tbest

def _rot_up(deg):
    a0, a1 = hax(); a = np.radians(deg); T = np.eye(4)
    T[a0, a0] = T[a1, a1] = np.cos(a); T[a0, a1] = -np.sin(a); T[a1, a0] = np.sin(a); return T

def _quick_polish(S, Tg, T, n=8000, iters=12, seed=0):
    """A few robust point-to-plane iterations (4-DoF) at short range: brings a roughly-right hypothesis onto the
    surfaces so that correct and wrong hypotheses become clearly separable when scored."""
    v = C["VOXEL"]; idx = np.arange(len(S["pts"]))
    if len(idx) > n: idx = np.random.default_rng(seed).choice(idx, n, replace=False)
    P0 = S["pts"][idx]
    for it in range(iters):
        dm = 6 * v if it < iters // 2 else 3 * v
        P = tf(P0, T); d, j = Tg["tree"].query(P, distance_upper_bound=dm, workers=-1); ok = np.isfinite(d)
        if ok.sum() < 30: break
        Pk, Q, N = P[ok], Tg["pts"][j[ok]], Tg["nrm"][j[ok]]
        r = np.einsum("ij,ij->i", Pk - Q, N); c = dm / 2
        w = np.where(np.abs(r) < c, (1 - (r / c) ** 2) ** 2, 0.0)
        dT, _ = _solve_p2plane(Pk, Q, N, w, 4); T = dT @ T
    return T

def sym_structure_score(S, Tg, T):
    """Structure agreement in BOTH directions: a wrong alignment that parks the source on a dense corridor of the
    target scores high one way but low the other way."""
    return 0.5 * (structure_score(S, Tg, T) + structure_score(Tg, S, inv(T)))

def reg_ours_global(S, Tg, seed=0, return_all=False, prior=None):
    """Pool hypotheses from BEV-XC (correspondence-free) and RANSAC-4DoF (feature-based). Every hypothesis is polished
    briefly and scored by symmetric wall agreement; the best wins. The two generators fail for different reasons,
    and verification after polishing separates right from wrong far better than scoring the raw hypotheses.
    Selection = symmetric wall agreement x sqrt(feature agreement): smoke test showed wall agreement alone can prefer
    a symmetric look-alike produced by BEV-XC."""
    _, dbg = reg_bev(S, Tg, return_debug=True, prior=prior)
    _, hr = reg_ransac4(S, Tg, seed=seed, return_hyps=True, prior=prior)
    nb = 3 if prior is None else 6                                    # with a prior, keep more (all are plausible)
    cands = [h[0] for h in sorted(dbg["hyps"], key=lambda h: -h[1])[:nb]] + hr[:(5 if prior is None else 8)]
    pol = [_quick_polish(S, Tg, T) for T in cands]
    st = np.array([sym_structure_score(S, Tg, T) for T in pol]); fe = np.array([feature_score(S, Tg, T) for T in pol])
    # two independent cues must agree: geometry (walls overlap) x appearance (local-shape matches agree)
    sc = st * np.sqrt(fe + 1e-3)
    if prior is not None:                                             # polishing must not have walked out of the prior
        cs = S["pts"].mean(0); ax_ = hax()
        inside = np.array([_prior_ok(*tf(cs[None], T)[0][ax_], (prior[0], prior[1] + 0.5)) for T in pol])
        if inside.any(): sc = np.where(inside, sc, -np.inf)
    best = int(np.argmax(sc))
    return (pol[best], dict(structure=st, features=fe, source=("bev" if best < nb else "r4"))) if return_all else pol[best]

# ---------- one weighted point-to-plane solver for every refiner ----------
def _solve_p2plane(P, Q, N, w, dof):
    r = np.einsum("ij,ij->i", P - Q, N)
    if dof == 6:
        J = np.hstack([np.cross(P, N), N])
    else:   # yaw about UP + translation
        e = np.zeros(3); e[C["UP"]] = 1.0
        J = np.hstack([np.einsum("ij,ij->i", np.cross(e[None], P), N)[:, None], N])
    A = J.T @ (w[:, None] * J); b = -J.T @ (w * r)
    try: x = np.linalg.solve(A + 1e-9 * np.eye(len(A)), b)
    except np.linalg.LinAlgError: return np.eye(4), 0.0
    T = np.eye(4)
    if dof == 6:
        T[:3, :3] = o3d.geometry.get_rotation_matrix_from_axis_angle(x[:3]); T[:3, 3] = x[3:]
    else:
        T[:3, :3] = (rotz(np.degrees(x[0])) if C["UP"] == 2 else _rot_up(np.degrees(x[0])))[:3, :3]; T[:3, 3] = x[1:]
    return T, float(np.linalg.norm(x))

def refine(S, Tg, T0, mode="icp", dof=6, iters=None, mask=None, max_pts=30000, seed=0, return_info=False):
    """mode: icp (L2), tukey, trimmed, gnc (method B v2).
    All refiners share the same correspondence schedule (MAXD0 -> 2 voxel), so they differ ONLY in the robust weighting."""
    iters = iters or (90 if mode == "gnc" else 60)
    v = C["VOXEL"]; idx = np.arange(len(S["pts"])) if mask is None else np.flatnonzero(mask)
    if len(idx) > max_pts: idx = np.random.default_rng(seed).choice(idx, max_pts, replace=False)
    P0 = S["pts"][idx]; T = T0.copy()
    d0 = C["MAXD0"]
    if mode == "gnc":
        # adaptive basin: open the search radius only as wide as the current misalignment needs. A start that is
        # already right (median wall distance a few cm) must not be dragged away by changed geometry 1 m away.
        dd, _ = Tg["tree"].query(tf(S["pts"][S["wall"]][::5] if S["wall"].any() else P0[::5], T), workers=-1)
        d0 = float(np.clip(4 * np.median(dd), 4 * v, C["MAXD0"]))
    dmax = np.geomspace(d0, 2 * v, max(2, iters // 2)).tolist() + [2 * v] * (iters - max(2, iters // 2))
    cbar = 1.5 * v; mu = None; w_last = None
    for it in range(iters):
        P = tf(P0, T)
        d, j = Tg["tree"].query(P, distance_upper_bound=dmax[it], workers=-1)
        ok = np.isfinite(d)
        if ok.sum() < 30: break
        Pk, Q, N = P[ok], Tg["pts"][j[ok]], Tg["nrm"][j[ok]]
        r = np.einsum("ij,ij->i", Pk - Q, N)
        if mode == "icp":
            w = np.ones(len(r))
        elif mode == "tukey":
            c = max(dmax[it] / 2, 2 * v); w = np.where(np.abs(r) < c, (1 - (r / c) ** 2) ** 2, 0.0)
        elif mode == "trimmed":
            w = (np.abs(r) <= np.quantile(np.abs(r), 0.7)).astype(float)
        elif mode == "gnc":
            # Graduated Non-Convexity with Geman-McClure (Yang et al. 2020): start almost quadratic (wide basin),
            # shrink mu -> 1 so the cost becomes strongly robust (precise, ignores changed geometry)
            if mu is None: mu = max(1.0, 2 * (np.max(np.abs(r)) / cbar) ** 2)
            w = (mu * cbar ** 2 / (r ** 2 + mu * cbar ** 2)) ** 2
            mu = max(1.0, mu / C["GNC_DECAY"])
        else:
            raise KeyError(mode)
        dT, step = _solve_p2plane(Pk, Q, N, w, dof)
        T = dT @ T; w_last = (ok, w)
        if step < 1e-6 and it > 5 and (mode != "gnc" or mu <= 1.0): break
    if mode == "gnc" and C.get("GNC_POLISH6", True):
        # gravity prior for the basin, full 6-DoF for the last centimetres: scans of different stages are tilted
        # against each other by a few tenths of a degree, which a 4-DoF solution cannot absorb
        for _ in range(15):
            P = tf(P0, T); d, j = Tg["tree"].query(P, distance_upper_bound=2 * v, workers=-1); ok = np.isfinite(d)
            if ok.sum() < 30: break
            Pk, Q, N = P[ok], Tg["pts"][j[ok]], Tg["nrm"][j[ok]]
            r = np.einsum("ij,ij->i", Pk - Q, N)
            w = (cbar ** 2 / (r ** 2 + cbar ** 2)) ** 2
            dT, step = _solve_p2plane(Pk, Q, N, w, 6); T = dT @ T
            if step < 1e-6: break
    if return_info:
        return T, dict(iters=it + 1, inlier_frac=float(np.mean(w_last[1] > 0.5)) if w_last else 0.0)
    return T

def ref_status_rule(shift, f0, f1, max_shift=0.30):
    """gt_ok: GT already at the optimum (<5 cm). refined: robust ICP moved it <=30 cm without worsening the wall fit
    (the GT lies inside the ICP basin, so the local optimum is the better reference). unverified: anything else."""
    if shift < 0.05: return "gt_ok"
    if shift <= max_shift and f1 >= f0 - 0.01: return "refined"
    return "unverified"

def reference_T(S, Tg, T_gt, max_shift=0.30):
    """Audited reference: polish the benchmark GT with a robust 6-DoF ICP (Open3D, Cauchy kernel -- a different
    implementation from every evaluated method) and keep the polish if it is small and does not worsen the fit.
    status: see ref_status_rule. 'unverified' pairs keep the GT and are excluded from FINE accuracy metrics."""
    v = C["VOXEL"]; T = T_gt.copy()
    for dmax in (0.30, 0.15, 2 * v):
        est = REG.TransformationEstimationPointToPlane(REG.CauchyLoss(k=dmax / 2))
        T = np.asarray(REG.registration_icp(S["pcd"], Tg["pcd"], dmax, T, est, REG.ICPConvergenceCriteria(max_iteration=30)).transformation).copy()
    shift = float(np.mean(np.linalg.norm(tf(S["pts"], T) - tf(S["pts"], T_gt), axis=1)))
    f0 = _fitness(S["pts"][S["wall"]], Tg, T_gt, 1.5 * v); f1 = _fitness(S["pts"][S["wall"]], Tg, T, 1.5 * v)
    status = ref_status_rule(shift, f0, f1, max_shift)
    Tr = T_gt if status == "unverified" else T
    dR = T[:3, :3] @ T_gt[:3, :3].T
    rot = float(np.degrees(np.arccos(np.clip((np.trace(dR) - 1) / 2, -1, 1))))
    return Tr, dict(ref_status=status, gt_shift_m=shift, gt_shift_deg=rot, fit_gt=f0, fit_ref=f1)

GLOBAL_METHODS = {
    "ransac6": lambda S, Tg: reg_ransac6(S, Tg),
    "fgr": lambda S, Tg: reg_fgr(S, Tg),
    "ransac4 (ours)": lambda S, Tg: reg_ransac4(S, Tg),
    "bev-xc (ours)": lambda S, Tg: reg_bev(S, Tg),
    "bev+r4 (ours)": lambda S, Tg: reg_ours_global(S, Tg),
}
REFINERS = {   # name: (mode, dof)
    "icp": ("icp", 6), "tukey": ("tukey", 6), "trimmed": ("trimmed", 6), "B v2: gnc+polish (ours)": ("gnc", 4),
}

# =====================================================================================
# 4. VISIBILITY-AWARE CHANGE DETECTION
# =====================================================================================
def _sph(Prel, res):
    r = np.linalg.norm(Prel, axis=1) + 1e-9
    a0, a1 = hax()
    az = np.arctan2(Prel[:, a1], Prel[:, a0]); el = np.arcsin(np.clip(Prel[:, C["UP"]] / r, -1, 1))
    W = int(round(360 / res)); H = int(round(180 / res))
    ix = np.clip(((az + np.pi) / (2 * np.pi) * W).astype(np.int64), 0, W - 1)
    iy = np.clip(((el + np.pi / 2) / np.pi * H).astype(np.int64), 0, H - 1)
    return r, iy, ix, (H, W)

def self_occlusion(P, o, res=0.5, n=40000, seed=0):
    """For a real scan, every point is the FIRST return along its ray. Measured from the true scanner position,
    almost no point hides behind another; from a wrong position, many do. Lower = more plausible origin."""
    if len(P) > n: P = P[np.random.default_rng(seed).choice(len(P), n, replace=False)]
    r, iy, ix, shp = _sph(P - o, res)
    D = np.full(shp, np.inf); np.minimum.at(D, (iy, ix), r)
    return float(np.mean(r > D[iy, ix] + 0.05 + 0.03 * r))

def estimate_origin(P, seed=0):
    """Scanner position = the viewpoint minimising self-occlusion (coarse-to-fine grid search)."""
    rng = np.random.default_rng(seed)
    Ps = P if len(P) <= 30000 else P[rng.choice(len(P), 30000, replace=False)]
    up, (a0, a1) = C["UP"], hax()
    z = Ps[:, up]; floor = np.percentile(z, 2)
    # TLS point density is highest right below the scanner: start there
    lowpts = Ps[z < floor + 0.3]
    if len(lowpts) > 100:
        H2, xe, ye = np.histogram2d(lowpts[:, a0], lowpts[:, a1], bins=40)
        i, j = np.unravel_index(np.argmax(ndimage.gaussian_filter(H2, 1)), H2.shape)
        c0 = np.array([(xe[i] + xe[i + 1]) / 2, (ye[j] + ye[j + 1]) / 2])
    else:
        c0 = np.median(Ps[:, [a0, a1]], 0)
    best = (np.inf, None)
    for span, step, zs in [(4.0, 0.5, floor + np.array([0.4, 0.9, 1.4, 1.9])), (0.6, 0.15, None), (0.15, 0.05, None)]:
        if zs is None: zs = best[1][up] + np.array([-0.3, -0.15, 0, 0.15, 0.3]) * (step / 0.15)
        cxy = c0 if best[1] is None else best[1][[a0, a1]]
        g = np.arange(-span, span + 1e-9, step)
        for dx in g:
            for dy in g:
                for zz in zs:
                    o = np.zeros(3); o[a0], o[a1], o[up] = cxy[0] + dx, cxy[1] + dy, zz
                    s = self_occlusion(Ps, o, res=1.0, n=8000)
                    if s < best[0]: best = (s, o)
    return best[1], self_occlusion(P, best[1])

def resolve_origins(keys, cloud, mode=None, verbose=True):
    """Decide each fragment's scanner position: local origin if it is self-consistent, else estimated."""
    mode = mode or C["ORIGIN_MODE"]; rows = []
    if not keys:
        return pd.DataFrame(columns=["origin_src", "self_occ_local", "self_occ_used", "ox", "oy", "oz"])
    for k in keys:
        P = read_points(INDEX[k]); s_loc = self_occlusion(P, np.zeros(3))
        o, s, src = np.zeros(3), s_loc, "local"
        if mode == "estimate" or (mode == "auto" and s_loc > 0.05):
            o_est, s_est = estimate_origin(P)
            if mode == "estimate" or s_est < s_loc: o, s, src = o_est, s_est, "estimated"
        cloud.origins[k] = o; cloud.d.pop(k, None)
        rows.append(dict(key=k, origin_src=src, self_occ_local=s_loc, self_occ_used=s, ox=o[0], oy=o[1], oz=o[2]))
        if verbose: print(f"  {k}: local={s_loc:.3f} -> {src} ({s:.3f})")
        elif len(rows) % 25 == 0: print(f"  origins {len(rows)}/{len(keys)}")
    return pd.DataFrame(rows).set_index("key")

class DepthMap:
    """Spherical (equirectangular) range image of one scan seen from its scanner: D[pixel] = nearest return."""
    def __init__(self, P_local, origin, res=None, max_pts=None, seed=0):
        res = res or C["DEPTH_RES"]; max_pts = max_pts or C["DEPTH_MAX_PTS"]
        if len(P_local) > max_pts: P_local = P_local[np.random.default_rng(seed).choice(len(P_local), max_pts, replace=False)]
        if res == "auto":   # ~4 points per pixel on average: sparse fragments need coarser pixels or every pixel is empty
            res = float(np.clip(np.sqrt(4 * 41253.0 / max(len(P_local), 1)), 0.2, 3.0))
        self.o = np.asarray(origin, float); self.res = res
        r, iy, ix, shp = _sph(P_local - self.o, res)
        D = np.full(shp, np.inf, np.float32); np.minimum.at(D, (iy, ix), r.astype(np.float32))
        self.D = D
        self.Dmin = ndimage.minimum_filter(D, size=3, mode=["nearest", "wrap"])     # conservative: nearest in 3x3
        self.n = ndimage.uniform_filter(np.isfinite(D).astype(np.float32), size=3, mode=["nearest", "wrap"]) * 9
    def query(self, P_local):
        r, iy, ix, _ = _sph(P_local - self.o, self.res)
        return r, self.Dmin[iy, ix], self.n[iy, ix]
    def free(self, P_local, tau, margin_rel, min_returns=3):
        """True where this scanner measured a return BEYOND the point = it saw empty space there."""
        r, Dm, nret = self.query(P_local)
        return (nret >= min_returns) & (Dm > r + tau + margin_rel * r)

class NormalFreeSpace:
    """Fallback when a fragment has NO single scanner position (e.g. fused from a moving sensor).
    Every observed surface point q with normal n (oriented toward where the sensor was) certifies a short
    free-space tube in front of it: q + s*n for tau < s < L. A point inside such a tube was seen to be empty.
    Weaker than ray casting (only short range) but needs no viewpoint -- like the truncation band of a TSDF."""
    def __init__(self, entry, L=1.0, rho=None, k=24):
        self.e = entry; self.L = L; self.rho = rho or 2 * C["VOXEL"]; self.k = k
    def free(self, P_local, tau, margin_rel=0.0, min_returns=3):
        d, j = self.e["tree"].query(P_local, k=self.k, distance_upper_bound=self.L, workers=-1)
        ok = np.isfinite(d); j = np.where(ok, j, 0)
        D = P_local[:, None, :] - self.e["pts"][j]
        s_ = np.einsum("ijk,ijk->ij", D, self.e["nrm"][j])
        lat2 = np.einsum("ijk,ijk->ij", D, D) - s_ ** 2
        hit = ok & (s_ > tau) & (s_ < self.L) & (lat2 < self.rho ** 2)
        return hit.sum(1) >= min_returns

def visibility_model(key, cloud, origins, origin_table=None, max_selfocc=0.05):
    """Ray-casting depth map if the fragment has a credible single scanner position, else the normal fallback."""
    so = None
    if origin_table is not None and key in origin_table.index: so = float(origin_table.self_occ_used[key])
    if so is not None and so > max_selfocc:
        return NormalFreeSpace(cloud(key))
    return DepthMap(read_points(INDEX[key]), origins.get(key, np.zeros(3)))

def p2plane_dist(P, other):
    d, j = other["tree"].query(P, workers=-1)
    r = np.abs(np.einsum("ij,ij->i", P - other["pts"][j], other["nrm"][j]))
    return np.where(d > 3 * C["VOXEL"], d, r)

UNCHANGED, CHANGED, UNOBSERVED = 0, 1, 2
def classify(P, other, views, tau=None, margin_rel=None, min_returns=3):
    """Three-class change detection for points P (already in the frame of `other`).
       UNCHANGED  : a surface of the other epoch lies within tau (point-to-plane)
       CHANGED    : no such surface AND some scanner of the other epoch looked THROUGH this spot (free-space violation)
       UNOBSERVED : otherwise (occluded or out of view) -> we abstain instead of guessing.
    views: list of (DepthMap, T_frame_to_scan) for the other epoch's scanners."""
    tau = tau or C["TAU"]; mr = C["FS_MARGIN_REL"] if margin_rel is None else margin_rel
    d = p2plane_dist(P, other)
    lab = np.full(len(P), UNOBSERVED, np.int8); lab[d < tau] = UNCHANGED
    todo = np.flatnonzero(lab != UNCHANGED)
    for dm, T in views:
        if len(todo) == 0: break
        free = dm.free(tf(P[todo], T), tau, mr, min_returns)
        lab[todo[free]] = CHANGED
        todo = todo[~free]
    return lab, d

def classify_baseline(P, other, tau=None):
    tau = tau or C["TAU"]; d = p2plane_dist(P, other)
    return np.where(d < tau, UNCHANGED, CHANGED).astype(np.int8), d

def covisible(P, other_pts, cell=None):
    """Eroded XY co-coverage mask (same region definition for baseline and ours, so metrics compare like with like)."""
    cell = cell or 2 * C["FOOT_CELL"]; ax = hax()
    fp = footprint(other_pts, cell)
    inner = {c for c in fp if all((c[0] + i, c[1] + j) in fp for i in (-1, 0, 1) for j in (-1, 0, 1))}
    return np.array([tuple(c) in inner for c in np.floor(P[:, ax] / cell).astype(np.int64)])

# ---------- physically consistent object injection ----------
def _ray_box(o, P, lo, hi):
    """Entry/exit distances of rays o->P with an axis-aligned box (slab method)."""
    d = P - o; L = np.linalg.norm(d, axis=1) + 1e-12; u = d / L[:, None]
    with np.errstate(divide="ignore", invalid="ignore"):
        t1 = (lo - o) / u; t2 = (hi - o) / u
    tmin = np.nanmax(np.minimum(t1, t2), axis=1); tmax = np.nanmin(np.maximum(t1, t2), axis=1)
    return tmin, tmax, L

def box_surface(lo, hi, spacing, rng):
    pts, nrm = [], []
    for ax in range(3):
        for side, val in ((-1, lo[ax]), (1, hi[ax])):
            if ax == C["UP"] and side == -1: continue             # resting on the floor
            o_ = [a for a in range(3) if a != ax]
            n = max(20, int(np.prod(hi[o_] - lo[o_]) / spacing ** 2))
            X = np.zeros((n, 3)); X[:, ax] = val
            for a in o_: X[:, a] = rng.uniform(lo[a], hi[a], n)
            N = np.zeros((n, 3)); N[:, ax] = side
            pts.append(X); nrm.append(N)
    return np.vstack(pts), np.vstack(nrm)

def inject_object(P_raw, origin, lo, hi, rng, spacing=0.01):
    """Insert a box into a scan AS THE SCANNER WOULD SEE IT: only faces visible from the scanner are added and
    every existing return the box would hide is removed. Returns (new_raw_points, object_points)."""
    tmin, tmax, L = _ray_box(origin, P_raw, lo, hi)
    hidden = (tmax > tmin) & (tmin > 0) & (tmin < L - 0.01)
    S, _ = box_surface(lo, hi, spacing, rng)
    smin, smax, SL = _ray_box(origin, S, lo, hi)
    vis = np.abs(smin - SL) < 0.02
    obj = S[vis]
    return np.vstack([P_raw[~hidden], obj]), obj

# =====================================================================================
# 5. BUILDING-SCALE EVIDENCE FUSION
# =====================================================================================
def stage_map(keys, W, cloud, voxel=None):
    """All fragments of one stage in the common frame -> one voxelised map entry.
    Normals keep the per-scan orientation (towards the scanner that saw the point): FPFH features depend on normal
    direction, so re-estimating unoriented normals on the merged map breaks scan-to-map feature matching."""
    P = np.vstack([tf(cloud(k)["pts"], W[k]) for k in keys])
    N = np.vstack([cloud(k)["nrm"] @ W[k][:3, :3].T for k in keys])
    pcd = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(P)); pcd.normals = o3d.utility.Vector3dVector(N)
    pcd = pcd.voxel_down_sample(voxel or C["VOXEL"])                  # averages the normals inside each voxel
    Nm = np.asarray(pcd.normals).copy()
    est = o3d.geometry.PointCloud(pcd); est.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=3 * C["VOXEL"], max_nn=30))
    Ne = np.asarray(est.normals)
    # clean local geometry from re-estimation, sign taken from the scanner-oriented average
    # (where opposite-facing normals cancelled out, ln ~ 0, the sign stays arbitrary: rare, thin walls only)
    Ne = Ne * np.where(np.einsum("ij,ij->i", Ne, Nm) < 0, -1.0, 1.0)[:, None]
    pcd.normals = o3d.utility.Vector3dVector(Ne)
    return finish_entry(pcd)

def fused_classify(P_world, other_map, views_world, tau=None, max_range=30.0):
    """Every scanner of the other epoch votes. Unchanged if any surface agrees; changed if any scanner saw through."""
    tau = tau or C["TAU"]
    lab, d = classify(P_world, other_map, [], tau)          # consistency only
    todo = np.flatnonzero(lab != UNCHANGED)
    for dm, Tw2s, ow in views_world:
        if len(todo) == 0: break
        near = np.linalg.norm(P_world[todo] - ow, axis=1) < max_range
        sub = todo[near]
        if len(sub) == 0: continue
        free = dm.free(tf(P_world[sub], Tw2s), tau, C["FS_MARGIN_REL"])
        lab[sub[free]] = CHANGED
        todo = np.setdiff1d(todo, sub[free], assume_unique=True)
    return lab, d

def floor_levels(e, scanner_z=None, bin_=0.05):
    """Storey floor heights. With scanner heights (recommended): for each group of scanners at a similar height,
    the floor is the strongest horizontal surface 0.5-3 m below them. Without: lowest strong horizontal peak."""
    up = C["UP"]; z = e["pts"][e["horiz"], up]
    if len(z) < 100: z = e["pts"][:, up]
    h, b = np.histogram(z, np.arange(z.min() - bin_, z.max() + 2 * bin_, bin_)); mid = (b[:-1] + b[1:]) / 2
    hs = ndimage.gaussian_filter1d(h.astype(float), 1)
    if scanner_z is None or len(scanner_z) == 0:
        pk = [i for i in range(1, len(hs) - 1) if hs[i] >= hs[i - 1] and hs[i] >= hs[i + 1] and h[i] > 0.02 * len(z)]
        return [float(mid[min(pk)])] if pk else [float(np.percentile(z, 2))]
    groups = []
    for zz in np.sort(np.asarray(scanner_z)):
        if not groups or zz - groups[-1][-1] > 1.5: groups.append([zz])
        else: groups[-1].append(zz)
    out = []
    for g in groups:
        zs = np.median(g); m = (mid < zs - 0.5) & (mid > zs - 3.0)
        if m.any(): out.append(float(mid[m][np.argmax(hs[m])]))
    return out or [float(np.percentile(z, 2))]

# =====================================================================================
# 5b. ONE-CALL DATA PREPARATION (shared by every notebook, cached)
# =====================================================================================
PREP_VERSION = 3      # bump whenever the GT logic changes, so stale caches are never reused

def prepare(data_root, extract_dir, work, building, overrides=None, prep_from=None, min_ov=0.3, min_geom=0.10,
            origins=True, verbose=True):
    """Load NSS, verify the GT convention, chain poses, resolve scanner origins, generate same/cross-stage pairs.
    Cached in <work>/nss_prep/prep.pkl; pass prep_from=<folder with prep.pkl> to reuse a teammate's run."""
    import pickle, shutil
    pdir = Path(work) / "nss_prep"; pdir.mkdir(parents=True, exist_ok=True); f = pdir / "prep.pkl"
    def _bldg(path):
        try: return pickle.loads((Path(path) / "prep.pkl").read_bytes()).get("building")
        except Exception: return None
    if prep_from and (Path(prep_from) / "prep.pkl").exists() and _bldg(prep_from) != building:
        print(f"!! PREP_FROM holds a prep for {_bldg(prep_from)}, not {building}: ignored"); prep_from = None
    if (not prep_from or not (Path(prep_from) / "prep.pkl").exists()):
        hits = sorted(Path(data_root).rglob("prep.pkl")) if Path(data_root).exists() else []
        for h in hits:                       # take the first prep.pkl made for THIS building (several may be attached)
            if _bldg(h.parent) == building:
                prep_from = h.parent; print("found prep.pkl for", building, "automatically in", prep_from); break
        else:
            if hits: print(f"!! {len(hits)} prep.pkl attached, none made for {building}: the pair list will be rebuilt "
                           "(its order, and therefore the 240 selected pairs, can differ from other runs)")
    if prep_from and (Path(prep_from) / "prep.pkl").exists() and not f.exists():
        shutil.copy(Path(prep_from) / "prep.pkl", f); print("using prep from", prep_from)
    D = load_nss(data_root, extract_dir, overrides, verbose=verbose)
    cloud = CloudCache()
    if f.exists():
        R = pickle.loads(f.read_bytes())
        if R.get("building") == building and R.get("voxel") == C["VOXEL"] and R.get("version") == PREP_VERSION:
            cloud.origins = R["origins"]
            if verbose: print(f"loaded cached prep: {len(R['pairs'])} pairs, {len(R['W'])} posed fragments")
            R.update(D=D, cloud=cloud); return R
    t0 = time.time()
    conv, fit, conv_table = check_convention(D, building, cloud, verbose=verbose)
    W, comp, info = world_poses(D, building, conv, cloud, verbose)
    if verbose: print("pose info:", info)
    org = None
    if origins:
        if verbose: print("resolving scanner origins ...")
        org = resolve_origins(list(W), cloud, verbose=False)
        if verbose: print(org.origin_src.value_counts().to_dict(), "| median self-occlusion used:", round(org.self_occ_used.median(), 4))
    pairs = make_pairs(D, building, W, comp, cloud, min_ov=min_ov, min_geom=min_geom, verbose=verbose)
    R = dict(version=PREP_VERSION, building=building, voxel=C["VOXEL"], edges=D.get("edges_scored"), conv=conv, conv_fit=fit, conv_table=conv_table, W=W, comp=comp,
             pose_info=info, origins=dict(cloud.origins), origin_table=org, pairs=pairs, minutes=(time.time() - t0) / 60)
    f.write_bytes(pickle.dumps(R))
    if verbose: print(f"prep done in {R['minutes']:.1f} min -> {f}")
    R.update(D=D, cloud=cloud); return R

def scanner_world(R, key):
    """Scanner position of a fragment in the common frame."""
    return tf(R["origins"].get(key, np.zeros(3))[None], R["W"][key])[0]

# =====================================================================================
# 6. PLOTTING HELPERS
# =====================================================================================
LAB_COLORS = {UNCHANGED: "#b9bec7", CHANGED: "#d62728", UNOBSERVED: "#7fb3d5"}
METHOD_COLORS = {"ransac6": "#8c8c8c", "fgr": "#c7a76c", "ransac4 (ours)": "#6aa84f", "bev-xc (ours)": "#3d85c6",
                 "bev+r4 (ours)": "#1b5e20", "icp": "#4c78a8", "tukey": "#e45756", "trimmed": "#f58518",
                 "B v2: gnc+polish (ours)": "#2ca02c", "GT": "#222222"}

def topdown(ax, P, c, s=0.15, alpha=0.6, zband=None, title=None, rasterized=True):
    a0, a1 = hax(); P = np.asarray(P)
    if zband is not None:
        m = (P[:, C["UP"]] > zband[0]) & (P[:, C["UP"]] < zband[1]); P = P[m]
        c = np.asarray(c)[m] if not isinstance(c, str) and np.ndim(c) > 0 and len(np.atleast_1d(c)) == len(m) else c
    ax.scatter(P[:, a0], P[:, a1], c=c, s=s, alpha=alpha, linewidths=0, rasterized=rasterized)
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    if title: ax.set_title(title, fontsize=9)

def oblique(ax, P, colors, elev=35, azim=-60, s=0.3, n=120000, seed=0):
    """Fast pseudo-3D view: rotate, painter-sort by depth, 2D scatter (much faster than mplot3d)."""
    P = np.asarray(P); colors = np.asarray(colors)
    if len(P) > n:
        k = np.random.default_rng(seed).choice(len(P), n, replace=False); P, colors = P[k], colors[k]
    a, e = np.radians(azim), np.radians(elev)
    Rz = np.array([[np.cos(a), -np.sin(a), 0], [np.sin(a), np.cos(a), 0], [0, 0, 1]])
    Rx = np.array([[1, 0, 0], [0, np.cos(e), -np.sin(e)], [0, np.sin(e), np.cos(e)]])
    X = (P - P.mean(0)) @ (Rx @ Rz).T
    o = np.argsort(X[:, 1])[::-1]
    ax.scatter(X[o, 0], X[o, 2], c=colors[o], s=s, linewidths=0, rasterized=True)
    ax.set_aspect("equal"); ax.axis("off")

def save(fig, path, dpi=170):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=dpi, bbox_inches="tight"); return path

class _Tee:
    """Copy everything printed into a log file as well (so the upload zip contains the full console output)."""
    def __init__(self, stream, path): self.s = stream; self.f = open(path, "a", buffering=1)
    def write(self, x):
        try: self.f.write(x)
        except Exception: pass
        return self.s.write(x)
    def flush(self):
        try: self.f.flush()
        except Exception: pass
        self.s.flush()
    def __getattr__(self, k): return getattr(self.s, k)

def start_log(out):
    import sys
    out = Path(out); out.mkdir(parents=True, exist_ok=True)
    base = sys.stdout.s if isinstance(sys.stdout, _Tee) else sys.stdout
    sys.stdout = _Tee(base, out / "console_log.txt")
    print(f"\n===== log started {time.strftime('%Y-%m-%d %H:%M:%S')} =====")

def bundle(work, nb, out, extra=()):
    """One zip with everything this notebook produced (figures, CSVs, JSON, console log) for sharing/verification."""
    import sys
    work, out = Path(work), Path(out)
    if isinstance(sys.stdout, _Tee): sys.stdout.flush()
    meta = dict(notebook=nb, finished=time.strftime("%Y-%m-%d %H:%M:%S"), config={k: v for k, v in C.items()})
    (out / "run_info.json").write_text(json.dumps(meta, indent=1, default=str))
    zpath = work / f"UPLOAD_{nb}.zip"
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        for f in sorted(out.rglob("*")):
            if f.is_file(): z.write(f, f.relative_to(out.parent))
        for f in extra:
            f = Path(f)
            if f.exists(): z.write(f, f"{nb}/extra/{f.name}")
    mb = zpath.stat().st_size / 1e6
    print(f"\n>>> UPLOAD THIS FILE: {zpath}  ({mb:.1f} MB)")
    print("    Kaggle: right panel -> Output -> /kaggle/working -> click the file to download.")
    return zpath

def append_csv(path, rows):
    path = Path(path); pd.DataFrame(rows).to_csv(path, mode="a", header=not path.exists(), index=False)
def done_keys(path, cols):
    path = Path(path)
    if not path.exists(): return set()
    d = pd.read_csv(path); return set(map(tuple, d[cols].astype(str).values))

In [ ]:
import os, json, time, shutil, importlib
from pathlib import Path
import numpy as np, pandas as pd, matplotlib, matplotlib.pyplot as plt
import nss_lib as L; importlib.reload(L)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": .25, "font.size": 10})

# ---------------- CONFIG (the only cell you normally edit) ----------------
DATA_ROOT   = Path(os.environ.get("NSS_ROOT", "/kaggle/input"))
EXTRACT_DIR = Path(os.environ.get("NSS_EXTRACT", "/tmp/nss_extract"))
WORK        = Path(os.environ.get("NSS_WORK", "/kaggle/working"))
PREP_FROM   = None          # e.g. "/kaggle/input/nss-prep"  (folder that contains prep.pkl from NB0)
BUILDING    = "Bldg3"
QUICK       = True          # True = small smoke test (~15-30 min). Set False for the real run (commit).
if "NSS_QUICK" in os.environ: QUICK = os.environ["NSS_QUICK"] == "1"
ANNOT_OVERRIDES = {}        # only if NB0 says the annotation format was not detected
L.configure(VOXEL=0.05, TAU=0.15, SEED=0)
print("QUICK =", QUICK)

In [ ]:
NB = "nb5_finetune"
OUT = WORK / "nss_out" / NB; FIG = OUT / "figures"; FIG.mkdir(parents=True, exist_ok=True)
L.start_log(OUT)
import subprocess, importlib, importlib.util, re, random
STUB = False
CPU_TEST = os.environ.get("NSS_PREDATOR_CPU_TEST") == "1"   # local testing only: real network on CPU, random weights
REPO = WORK / "OverlapPredator"
N_POINTS = 2500; RANSAC_DIST_SCALED = 0.05
# ---- fine-tuning settings ----
TRAIN_BUILDINGS = os.environ.get("NSS_FT_TRAIN", "Bldg1,Bldg2,Bldg6").split(",")
VAL_BUILDING    = os.environ.get("NSS_FT_VAL", "Bldg4")
SCALE           = 0.19078594207214963   # metres -> network units; NB4's frozen choice on Bldg3 dev pairs
PAIRS_PER_BLDG  = 8 if QUICK else 1000    # training pairs per building (half same-stage, half cross-stage)
N_VAL           = 6 if QUICK else 30
EPOCHS          = 1 if QUICK else 12
TIME_BUDGET_H   = 0.25 if QUICK else 10.0 # stop training after this many hours (Kaggle sessions end at 12 h)
LR, MOMENTUM, WEIGHT_DECAY, GAMMA = 5e-4, 0.98, 1e-6, 0.95
W_CIRCLE, W_OVERLAP, W_SALIENCY = 1.0, 1.0, 1.0
MAX_IN_TRAIN    = 20000                   # points per cloud during training
COARSE_TRAIN    = 800                     # coarse-level points per cloud during training (backward pass needs more memory)
print("train:", TRAIN_BUILDINGS, "| val:", VAL_BUILDING, "| test (never used here): Bldg3, Bldg5")

## 1. Install PREDATOR (GPU check, clone, weights, C++ helpers)
Each step prints what it did. **If a step fails, send the printed error.**

In [ ]:
import torch
if not STUB:
    if shutil.which("nvidia-smi"): print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout[:1200])
    print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
    assert torch.cuda.is_available() or CPU_TEST, "No GPU. Settings -> Accelerator -> GPU T4 x2"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "easydict", "pyyaml", "scikit-learn", "tensorboardX", "coloredlogs", "nibabel", "tqdm"], check=False)
    if not (REPO / "models").exists():
        r = subprocess.run(["git", "clone", "--depth", "1", "https://github.com/prs-eth/OverlapPredator.git", str(REPO)],
                           capture_output=True, text=True); print(r.stdout[-500:], r.stderr[-500:])
    assert (REPO / "models").exists(), "git clone failed: is Internet on?"
    W_PATH = REPO / "weights" / "indoor.pth"
    if not W_PATH.exists() and not CPU_TEST:
        hits = sorted(Path("/kaggle/input").rglob("indoor.pth")) if Path("/kaggle/input").exists() else []
        if hits:
            W_PATH.parent.mkdir(exist_ok=True); shutil.copy(hits[0], W_PATH); print("weights from input:", hits[0])
        else:
            z = REPO / "weights.zip"
            r = subprocess.run(["wget", "-q", "--no-check-certificate", "https://share.phys.ethz.ch/~gseg/Predator/weights.zip", "-O", str(z)],
                               capture_output=True, text=True); print("wget exit", r.returncode, r.stderr[-300:])
            subprocess.run(["unzip", "-o", "-q", str(z), "-d", str(REPO)])
            if not W_PATH.exists():
                hits = sorted(REPO.rglob("indoor.pth"))
                if hits: W_PATH.parent.mkdir(exist_ok=True); shutil.copy(hits[0], W_PATH)
    assert W_PATH.exists() or CPU_TEST, ("PREDATOR weights not found. Download https://share.phys.ethz.ch/~gseg/Predator/weights.zip "
                             "on a laptop, upload it as a Kaggle dataset, add it as input, and re-run this cell.")
    if W_PATH.exists(): print("weights:", W_PATH, f"{W_PATH.stat().st_size/1e6:.1f} MB")

In [ ]:
# C++ helpers (grid subsampling, radius neighbours). Their setup.py uses distutils, which Python 3.12+ removed, so we
# build with our own setuptools script. A pure-Python version is written next to them as a fallback: if the build
# fails (or gives different answers), the fallback is used automatically. Both do exactly the same job.
GRID_PY = '''import numpy as np
def subsample_batch(points, batches, features=None, classes=None, sampleDl=0.1, method="barycenters", max_p=0, verbose=0):
    points = np.asarray(points, np.float32); batches = np.asarray(batches).astype(np.int64).ravel()
    P_, L_, F_, C_ = [], [], [], []; s = 0
    for n in batches:
        n = int(n); P = points[s:s + n]
        if n == 0:
            L_.append(0); continue
        key = np.floor((P - P.min(0)) / sampleDl).astype(np.int64)
        _, inv, cnt = np.unique(key, axis=0, return_inverse=True, return_counts=True); inv = np.asarray(inv).ravel()
        m = len(cnt); S = np.zeros((m, 3)); np.add.at(S, inv, P); S /= cnt[:, None]
        P_.append(S.astype(np.float32)); L_.append(m)
        if features is not None:
            Fe = np.asarray(features, np.float32)[s:s + n]; G = np.zeros((m, Fe.shape[1])); np.add.at(G, inv, Fe)
            F_.append((G / cnt[:, None]).astype(np.float32))
        if classes is not None:
            lab = np.zeros(m, np.int32); lab[inv] = np.asarray(classes).astype(np.int32).ravel()[s:s + n]; C_.append(lab)
        s += n
    out = [np.concatenate(P_) if P_ else np.zeros((0, 3), np.float32), np.array(L_, np.int32)]
    if features is not None: out.append(np.concatenate(F_))
    if classes is not None: out.append(np.concatenate(C_))
    return tuple(out)
'''
NEIGH_PY = '''import numpy as np
from scipy.spatial import cKDTree
def batch_query(queries, supports, q_batches, s_batches, radius=0.1):
    Qa = np.asarray(queries, np.float64); Sa = np.asarray(supports, np.float64)
    qb = np.asarray(q_batches).astype(np.int64).ravel(); sb = np.asarray(s_batches).astype(np.int64).ravel()
    N = len(Sa); lists = []; mx = 0; q0 = s0 = 0
    for nq, ns in zip(qb, sb):
        nq, ns = int(nq), int(ns); Q = Qa[q0:q0 + nq]; S = Sa[s0:s0 + ns]
        if nq and ns:
            for i, l in enumerate(cKDTree(S).query_ball_point(Q, radius)):
                l = np.asarray(l, np.int64)
                if len(l): l = l[np.argsort(np.linalg.norm(S[l] - Q[i], axis=1), kind="stable")]
                lists.append(l + s0); mx = max(mx, len(l))
        else:
            lists.extend([np.zeros(0, np.int64)] * nq)
        q0 += nq; s0 += ns
    out = np.full((len(lists), mx), N, np.int64)
    for i, l in enumerate(lists): out[i, :len(l)] = l
    return out
'''
def build_and_check_cpp(repo):
    base = Path(repo).resolve() / "cpp_wrappers"; cloud_cpp = base / "cpp_utils" / "cloud" / "cloud.cpp"
    report = {}
    for sub, name, py in (("cpp_subsampling", "grid_subsampling", GRID_PY), ("cpp_neighbors", "radius_neighbors", NEIGH_PY)):
        d = base / sub
        (d / f"{name}.py").write_text(py)                      # fallback (extension modules take precedence)
        w = d / "wrapper.cpp"; t = w.read_text()                # NumPy 2 removed the old C-API names they use
        t = t.replace("NPY_IN_ARRAY", "NPY_ARRAY_IN_ARRAY")
        t = re.sub(r"PyArray_(NDIM|DIM|DATA)\((?!\(PyArrayObject)(\w+)", r"PyArray_\1((PyArrayObject*)\2", t)
        w.write_text(t)
        srcs = [str(cloud_cpp)] + [str(p) for p in sorted(d.rglob("*.cpp")) if "build" not in p.parts]
        (d / "setup_kaggle.py").write_text(
            "from setuptools import setup, Extension\nimport numpy\n"
            f"setup(name='{name}', ext_modules=[Extension('{name}', sources={srcs!r}, include_dirs=[numpy.get_include()], "
            "extra_compile_args=['-std=c++11', '-D_GLIBCXX_USE_CXX11_ABI=0'])])\n")
        r = subprocess.run([sys.executable, "setup_kaggle.py", "build_ext", "--inplace"], cwd=d, capture_output=True, text=True)
        report[sub] = "compiled" if r.returncode == 0 else "build failed -> Python fallback"
        if r.returncode: print(sub, "build error (fallback will be used):", r.stderr[-800:])
    # check compiled vs fallback on random data; drop the compiled one if it disagrees
    rng = np.random.default_rng(0); P = rng.uniform(0, 1, (3000, 3)).astype(np.float32); bl = np.array([1800, 1200], np.int32)
    for sub, name in (("cpp_subsampling", "grid_subsampling"), ("cpp_neighbors", "radius_neighbors")):
        d = base / sub; so = list(d.glob(f"{name}*.so"))
        if not so: continue
        try:
            spec = importlib.util.spec_from_file_location(name, so[0]); mc = importlib.util.module_from_spec(spec); spec.loader.exec_module(mc)
            spec = importlib.util.spec_from_file_location(name + "_p", d / f"{name}.py"); mp = importlib.util.module_from_spec(spec); spec.loader.exec_module(mp)
            if name == "grid_subsampling":
                a = mc.subsample_batch(P, bl, sampleDl=0.1, max_p=0, verbose=0); b = mp.subsample_batch(P, bl, sampleDl=0.1)
                ok = abs(int(a[1].sum()) - int(b[1].sum())) <= 0.05 * int(b[1].sum())
            else:
                a = mc.batch_query(P, P, bl, bl, radius=0.08); b = mp.batch_query(P, P, bl, bl, radius=0.08)
                ok = all(set(a[i][a[i] < len(P)]) == set(b[i][b[i] < len(P)]) for i in range(0, 3000, 97))
        except Exception as e:
            ok = False; print(sub, "compiled module failed the check:", e)
        if not ok:
            for f in so: f.unlink()
            report[sub] = "compiled but mismatched -> Python fallback"
    print("C++ helpers:", report)
    return report

if not STUB:
    CPP_REPORT = build_and_check_cpp(REPO)

## 2. Load the model and self-test on PREDATOR's own demo scans
Expect `SELF-TEST: 3/3 (or 2/3) recovered the known transform -> model OK`. **If it says NOT working, stop and send the output.**

In [ ]:
REG = o3d_reg = L.REG
import open3d as o3d
def ransac_on_features(ps, pt, fs, ft, dist, iters=50000):
    S = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(np.asarray(ps, np.float64)))
    T = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(np.asarray(pt, np.float64)))
    F1 = REG.Feature(); F1.data = np.asarray(fs, np.float64).T.copy()
    F2 = REG.Feature(); F2.data = np.asarray(ft, np.float64).T.copy()
    r = REG.registration_ransac_based_on_feature_matching(S, T, F1, F2, False, dist, REG.TransformationEstimationPointToPoint(False), 3,
        [REG.CorrespondenceCheckerBasedOnEdgeLength(0.9), REG.CorrespondenceCheckerBasedOnDistance(dist)],
        REG.RANSACConvergenceCriteria(iters, 0.999))
    return np.asarray(r.transformation).copy()

if not STUB:
    for _n, _t in (("int", int), ("float", float), ("bool", bool), ("object", object)):
        if _n not in np.__dict__: setattr(np, _n, _t)          # old aliases some legacy code still uses
    dl = REPO / "datasets" / "dataloader.py"; t = dl.read_text()
    for line in ("from datasets.indoor import IndoorDataset", "from datasets.kitti import KITTIDataset",
                 "from datasets.modelnet import get_train_datasets, get_test_datasets"):
        if line in t and f"try:\n    {line}" not in t:
            t = t.replace(line, f"try:\n    {line}\nexcept Exception as _e:\n    print('skipped import:', {line!r})")
    dl.write_text(t)
    sys.path.insert(0, str(REPO))
    for m in [k for k in list(sys.modules) if k.split(".")[0] in ("datasets", "models", "lib", "kernels", "cpp_wrappers")]:
        del sys.modules[m]                                       # Kaggle ships a different 'datasets' package
    from datasets.dataloader import get_dataloader
    from models.architectures import KPFCNN
    import yaml
    from easydict import EasyDict as edict
    docs = [d for d in yaml.safe_load_all(open(REPO / "configs" / "test" / "indoor.yaml")) if isinstance(d, dict)]
    cfg = {}
    for sec, val in docs[0].items():
        if isinstance(val, dict): cfg.update(val)
    cfg = edict(cfg); cfg.device = torch.device("cpu" if CPU_TEST else "cuda")
    arch = ["simple", "resnetb"]
    for _ in range(cfg.num_layers - 1): arch += ["resnetb_strided", "resnetb", "resnetb"]
    for _ in range(cfg.num_layers - 2): arch += ["nearest_upsample", "unary"]
    arch += ["nearest_upsample", "last_unary"]
    cfg.architecture = arch
    _cwd = os.getcwd(); os.chdir(REPO)                         # kernel point files are written relative to the repo
    try:
        model = KPFCNN(cfg).to(cfg.device)
        if CPU_TEST: print("CPU_TEST: random weights (plumbing test only)")
        else:
            state = torch.load(W_PATH, map_location=cfg.device, weights_only=False)
            model.load_state_dict(state["state_dict"])
        model.eval()
    finally:
        os.chdir(_cwd)
    print("model loaded:", sum(p.numel() for p in model.parameters()) / 1e6, "M parameters")
else:
    from easydict import EasyDict as edict
    cfg = edict(first_subsampling_dl=0.025)

class PairSet(torch.utils.data.Dataset):
    """Two (already voxelised, network-scale) clouds in exactly the tuple format PREDATOR's collate function expects."""
    def __init__(self, config, items): self.config = config; self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        s, t = self.items[i]
        return (s, t, np.ones_like(s[:, :1]).astype(np.float32), np.ones_like(t[:, :1]).astype(np.float32),
                np.eye(3).astype(np.float32), np.ones((3, 1)).astype(np.float32), torch.ones(1, 2).long(), s, t, torch.ones(1))

MAX_IN = 25000      # points per cloud fed to the network (3DMatch fragments are about this size; more runs out of GPU memory)
def voxel(P, dl, max_n=MAX_IN, seed=0):
    p = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(np.asarray(P, np.float64))).voxel_down_sample(dl)
    P = np.asarray(p.points, np.float32)
    if len(P) > max_n: P = P[np.random.default_rng(seed).choice(len(P), max_n, replace=False)]
    return P

NL = None
COARSE_MAX = 1500    # coarse-level points per cloud (memory ~ 256 x N^2 floats per attention layer)
CROP_LOG = []
def predator_infer(src, tgt):
    """src, tgt: network-scale voxelised clouds. Returns keypoints, features and overlap x saliency scores per cloud."""
    if STUB:     # FPFH stand-in so the rest of the notebook can be tested without the network
        out = []
        for P in (src, tgt):
            pc = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(P.astype(np.float64)))
            pc.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
            f = REG.compute_fpfh_feature(pc, o3d.geometry.KDTreeSearchParamHybrid(radius=0.15, max_nn=100))
            out.append((P, np.asarray(f.data).T.astype(np.float32), np.ones(len(P), np.float32)))
        return out[0][0], out[1][0], out[0][1], out[1][1], out[0][2], out[1][2]
    # PREDATOR's attention builds a [256, N, N] tensor on the coarsest level (N = coarse points per cloud). NSS scans are
    # much larger than 3DMatch fragments, so N can exceed what a 16 GB GPU holds. Crop around each cloud's centre until safe.
    for attempt in range(6):
        loader, _ = get_dataloader(PairSet(cfg, [(src, tgt)]), batch_size=1, num_workers=0, shuffle=False, neighborhood_limits=NL)
        inputs = next(iter(loader)); nc = [int(x) for x in inputs["stack_lengths"][-1]]
        if max(nc) <= COARSE_MAX: break
        crop = lambda P, n: P[np.argsort(np.linalg.norm((P - np.median(P, 0))[:, :2], axis=1))[:max(2000, int(len(P) * min(1.0, 0.85 * COARSE_MAX / n)))]]
        src = crop(src, nc[0]) if nc[0] > COARSE_MAX else src
        tgt = crop(tgt, nc[1]) if nc[1] > COARSE_MAX else tgt
        CROP_LOG.append((attempt, nc))
    for k, v in inputs.items():
        if isinstance(v, list): inputs[k] = [x.to(cfg.device) if torch.is_tensor(x) else x for x in v]
        elif torch.is_tensor(v): inputs[k] = v.to(cfg.device)
    with torch.no_grad():
        feats, s_ov, s_sal = model(inputs)
    pts = inputs["points"][0]; ls = int(inputs["stack_lengths"][0][0]); sc = (s_ov * s_sal)
    c = lambda x: x.detach().cpu().numpy()
    return c(pts[:ls]), c(pts[ls:]), c(feats[:ls]), c(feats[ls:]), c(sc[:ls]), c(sc[ls:])

def sample_by_score(P, F, s, n, rng):
    if len(P) <= n: return P, F
    p = (np.asarray(s, np.float64) + 1e-6); p /= p.sum()
    idx = rng.choice(len(P), n, replace=False, p=p); return P[idx], F[idx]

if not STUB:
    demo = [torch.load(REPO / "assets" / f, weights_only=False).astype(np.float32) for f in ("cloud_bin_21.pth", "cloud_bin_34.pth")]
    _, NL = get_dataloader(PairSet(cfg, [(demo[0], demo[1])]), batch_size=1, num_workers=0, shuffle=False)
    # Known-answer test: split one demo scan into two overlapping halves, move one by a known rotation + translation,
    # and check that PREDATOR + RANSAC recovers it. (The demo pair itself has low overlap and no ground truth.)
    P = demo[0]; x = P[:, 0]; lo_, hi_ = np.quantile(x, [0.65, 0.35])
    T_known = L.make_T(o3d.geometry.get_rotation_matrix_from_axis_angle(np.array([0.3, -0.2, 1.4])), np.array([0.8, -0.5, 0.2]))
    s_test, t_test = P[x < lo_], L.tf(P[x > hi_], T_known).astype(np.float32)
    rng = np.random.default_rng(0); res_ = []
    for trial in range(3):
        t0 = time.time(); ks, kt, fs, ft, ss, st = predator_infer(s_test, t_test)
        a, fa = sample_by_score(ks, fs, ss, 1000, rng); b, fb = sample_by_score(kt, ft, st, 1000, rng)
        e = L.reg_errors(ransac_on_features(a, b, fa, fb, RANSAC_DIST_SCALED), T_known, s_test); res_.append(e)
        print(f"self-test {trial+1}: rotation error {e['rre']:.1f} deg, translation error {e['rte']*100:.1f} cm ({time.time()-t0:.1f} s)")
    n_ok = sum(e["rre"] < 10 and e["rte"] < 0.3 for e in res_)      # coarse (no ICP here); random weights give 80-180 deg
    print(f"SELF-TEST: {n_ok}/3 recovered the known transform", "-> model OK" if n_ok >= 2 else
          "-> !! the model is NOT working: stop and send this output")

## 3. Training and validation pairs (labels checked before use)

In [ ]:
D = L.load_nss(DATA_ROOT, EXTRACT_DIR, ANNOT_OVERRIDES)
frags = D["frags"]; pa = D["pairs_annot"].copy()
pa = pa[pa.T_raw.notna()].copy()
pa["building"] = pa.src.map(frags.building)
pa["cross"] = pa.src.map(frags.stage_num) != pa.tgt.map(frags.stage_num)
pa["key"] = [tuple(sorted(k)) for k in zip(pa.src, pa.tgt)]
pa = pa.drop_duplicates("key")
if pa.overlap_annot.notna().any(): pa = pa[pa.overlap_annot.fillna(0) >= 0.3]
cloud = L.CloudCache(max_items=60)
CONV = {}
for b in TRAIN_BUILDINGS + [VAL_BUILDING]:
    conv, fit, _ = L.check_convention(D, b, cloud, n=20, verbose=False)
    CONV[b] = conv; print(f"{b}: transform convention {conv!r} (median wall fit {fit:.2f}), annotated pairs {int((pa.building == b).sum())}")

def pick(b, n, seed):
    p = pa[pa.building == b].sample(frac=1, random_state=seed)
    return pd.concat([p[~p.cross].head(n // 2), p[p.cross].head(n - n // 2)])

def T_of(r):
    return r.T_raw if CONV[r.building] == "pair_T" else L.inv(r.T_raw)

def check_label(args):
    """Same-stage: NSS transform is exact (pure scanner-position difference). Cross-stage: polish with robust ICP and keep
    only verified pairs (the official cross-stage tilt is off by about 1 degree; see GT_FINDINGS_UPDATE.md)."""
    src, tgt, T, cross = args
    if not cross: return T, "same_stage"
    try:
        Tr, info = L.reference_T(cloud(src), cloud(tgt), T)
        return Tr, info["ref_status"]
    except Exception as e:
        return T, "error"

LIST_PKL = OUT / "train_pairs.pkl"
if not LIST_PKL.exists():
    rows = []
    for b, split in [(b_, "train") for b_ in TRAIN_BUILDINGS] + [(VAL_BUILDING, "val")]:
        if CONV[b] not in ("pair_T", "pair_T_inv"):
            print("!! skipping", b, "(transform convention not recognised)"); continue
        p = pick(b, N_VAL if split == "val" else PAIRS_PER_BLDG, seed=0 if split == "train" else 1)
        t0 = time.time()
        for i, r in enumerate(p.itertuples()):
            T, st = check_label((r.src, r.tgt, T_of(r), bool(r.cross)))
            rows.append(dict(src=r.src, tgt=r.tgt, T=T, cross=bool(r.cross), building=b, status=st, split=split))
            if i % 200 == 0: print(f"  {b}: labelled {i}/{len(p)} ({time.time()-t0:.0f}s)")
        print(f"{b}: {len(p)} pairs labelled in {time.time()-t0:.0f}s")
    lab = pd.DataFrame(rows); lab.to_pickle(LIST_PKL)
lab = pd.read_pickle(LIST_PKL)
print(lab.groupby(["split", "building", "status"]).size().to_string())
lab = lab[lab.status.isin(["same_stage", "gt_ok", "refined"])].reset_index(drop=True)
train_rows = lab[lab.split == "train"].reset_index(drop=True); val_rows = lab[lab.split == "val"].reset_index(drop=True)
print(f"-> {len(train_rows)} training pairs, {len(val_rows)} validation pairs")

## 4. Training data (augmentation, overlap labels, memory-safe crops)

In [ ]:
from scipy.spatial import cKDTree
from scipy.spatial.transform import Rotation as Rot
_pts = {}
def pts_of(k):
    if k not in _pts:
        if len(_pts) > 200: _pts.clear()
        _pts[k] = voxel(L.read_points(L.INDEX[k]) * SCALE, cfg.first_subsampling_dl, max_n=10**9)
    return _pts[k]

def coarse_count(P):
    return len(np.unique(np.floor(P / (cfg.first_subsampling_dl * 8)).astype(np.int64), axis=0))

class TrainSet(torch.utils.data.Dataset):
    """NSS pairs in PREDATOR's tuple format, in network units. Random heading (+ small tilt) on the source, light noise,
    crops around the overlap so the coarse level fits in memory, overlap labels from the checked transform."""
    def __init__(self, config, rows, augment=True, seed=0):
        self.config = config; self.rows = rows; self.augment = augment; self.rng = np.random.default_rng(seed)
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        for attempt in range(5):
            r = self.rows.iloc[(i + attempt * 7919) % len(self.rows)]   # a Series: use r["T"] (r.T would transpose)
            out = self.make(r)
            if out is not None: return out
        return self.make(r, force=True)
    def make(self, r, force=False):
        rng = self.rng
        S, Tq = pts_of(r["src"]).copy(), pts_of(r["tgt"]).copy()
        T = np.array(r["T"], float).copy(); T[:3, 3] *= SCALE                       # network units
        if self.augment:
            A = np.eye(4); A[:3, :3] = Rot.from_euler("zyx", [rng.uniform(0, 360), rng.normal(0, 1), rng.normal(0, 1)], degrees=True).as_matrix()
            c = S.mean(0); A[:3, 3] = c - A[:3, :3] @ c
            S = L.tf(S, A).astype(np.float32); T = T @ L.inv(A)
        St = L.tf(S, T)
        d, j = cKDTree(Tq).query(St, distance_upper_bound=self.config.overlap_radius)
        ok = np.isfinite(d)
        if ok.sum() < 50 and not force: return None
        # crop both clouds around a random overlap point until the coarse level is small enough
        ctr = St[rng.choice(np.flatnonzero(ok))] if ok.any() else Tq.mean(0)
        rad = 3.0 * SCALE * 4
        for _ in range(8):
            ks = np.linalg.norm((St - ctr)[:, :2], axis=1) < rad; kt = np.linalg.norm((Tq - ctr)[:, :2], axis=1) < rad
            if max(coarse_count(S[ks]), coarse_count(Tq[kt])) <= COARSE_TRAIN: break
            rad *= 0.8
        S, St, Tq = S[ks], St[ks], Tq[kt]
        if len(S) > MAX_IN_TRAIN: k = rng.choice(len(S), MAX_IN_TRAIN, replace=False); S, St = S[k], St[k]
        if len(Tq) > MAX_IN_TRAIN: Tq = Tq[rng.choice(len(Tq), MAX_IN_TRAIN, replace=False)]
        if self.augment:
            S = S + (rng.random(S.shape) - 0.5).astype(np.float32) * 0.005; Tq = Tq + (rng.random(Tq.shape) - 0.5).astype(np.float32) * 0.005
        St = L.tf(S, T)
        tree_t, tree_s = cKDTree(Tq), cKDTree(St)
        d1, j1 = tree_t.query(St, distance_upper_bound=self.config.overlap_radius)
        d2, j2 = tree_s.query(Tq, distance_upper_bound=self.config.overlap_radius)
        c1 = np.c_[np.flatnonzero(np.isfinite(d1)), j1[np.isfinite(d1)]]
        c2 = np.c_[j2[np.isfinite(d2)], np.flatnonzero(np.isfinite(d2))]
        corr = np.unique(np.vstack([c1, c2]), axis=0)
        if len(corr) < 50 and not force: return None
        if len(corr) == 0: corr = np.zeros((1, 2), np.int64)
        S = S.astype(np.float32); Tq = Tq.astype(np.float32)
        return (S, Tq, np.ones_like(S[:, :1]), np.ones_like(Tq[:, :1]), T[:3, :3].astype(np.float32),
                T[:3, 3:4].astype(np.float32), torch.from_numpy(corr.astype(np.int64)), S, Tq, torch.ones(1))

# loss settings from PREDATOR's own training config
docs_tr = [d for d in yaml.safe_load_all(open(REPO / "configs" / "train" / "indoor.yaml")) if isinstance(d, dict)]
for sec in ("loss",):
    for k, v in docs_tr[0][sec].items(): cfg[k] = v
lp = REPO / "lib" / "loss.py"; t = lp.read_text()
if "torch.device('cuda')" in t: lp.write_text(t.replace("torch.device('cuda')", "scores_overlap.device"))   # also works on CPU
for m in [k for k in list(sys.modules) if k.startswith("lib.loss")]: del sys.modules[m]
from lib.loss import MetricLoss
desc_loss = MetricLoss(cfg, log_scale=cfg.log_scale)
print("loss radii (network units): overlap", cfg.overlap_radius, "| positive", cfg.pos_radius, "| matchability", cfg.matchability_radius)

train_set = TrainSet(cfg, train_rows, augment=True, seed=0)
train_loader, NL = get_dataloader(train_set, batch_size=1, num_workers=0 if CPU_TEST else 2, shuffle=True)
print("neighbourhood limits (training):", NL)

## 5. Validation: registration success on held-out pairs (random heading)

In [ ]:
VAL_ITEMS = []
_vr = np.random.default_rng(5)
for r in val_rows.itertuples():
    S, Tq = pts_of(r.src), pts_of(r.tgt)
    A = np.eye(4); A[:3, :3] = Rot.from_euler("z", _vr.uniform(0, 360), degrees=True).as_matrix()
    c = S.mean(0); A[:3, 3] = c - A[:3, :3] @ c
    Tn = np.array(r.T, float).copy(); Tn[:3, 3] *= SCALE; Tn = Tn @ L.inv(A)
    VAL_ITEMS.append((L.tf(S, A).astype(np.float32), Tq, Tn, bool(r.cross)))

def validate():
    model.eval(); res = []; rng = np.random.default_rng(0)
    for Sv, Tv, Tn, cross in VAL_ITEMS:
        try:
            ks, kt, fs, ft, ss, st = predator_infer(Sv, Tv)
            a, fa = sample_by_score(ks, fs, ss, N_POINTS, rng); b, fb = sample_by_score(kt, ft, st, N_POINTS, rng)
            Te = ransac_on_features(a, b, fa, fb, RANSAC_DIST_SCALED)
        except RuntimeError as e:
            if torch.cuda.is_available(): torch.cuda.empty_cache()
            Te = np.eye(4)
        Te_m = Te.copy(); Te_m[:3, 3] /= SCALE; Tn_m = Tn.copy(); Tn_m[:3, 3] /= SCALE
        e = L.reg_errors(Te_m, Tn_m, Sv / SCALE); res.append(dict(cross=cross, **e))
    model.train(); r = pd.DataFrame(res)
    return float(r.succ_coarse.mean()), float(r[r.cross].succ_coarse.mean()) if r.cross.any() else np.nan

## 6. Fine-tune (time-boxed; best epoch kept)

In [ ]:
CKPT = WORK / "ft_last.pth"; BEST = OUT / "predator_nss_ft.pth"; HIST = OUT / "training_log.csv"
opt = torch.optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.ExponentialLR(opt, gamma=GAMMA)
start_epoch, best = 0, -1.0
if CKPT.exists():
    st_ = torch.load(CKPT, map_location=cfg.device, weights_only=False)
    model.load_state_dict(st_["state_dict"]); opt.load_state_dict(st_["opt"]); sched.load_state_dict(st_["sched"])
    start_epoch, best = st_["epoch"] + 1, st_["best"]; print("resumed from epoch", st_["epoch"])
hist = pd.read_csv(HIST).to_dict("records") if HIST.exists() else []
if start_epoch == 0 and not hist:
    v_all, v_cross = validate(); hist.append(dict(epoch=0, iters=0, loss=np.nan, val_succ=v_all, val_succ_cross=v_cross))
    print(f"epoch 0 (pretrained): validation success {v_all:.0%} (cross-stage {v_cross:.0%})"); best = v_all
    torch.save(dict(state_dict=model.state_dict(), scale=SCALE, NL=np.asarray(NL).tolist(), epoch=0, val_succ=v_all), BEST)
t_start = time.time(); model.train(); stop = False
for epoch in range(start_epoch + 1, EPOCHS + 1):
    run, n_ok, n_skip, t_ep = [], 0, 0, time.time()
    for it, inputs in enumerate(train_loader):
        for k, v in inputs.items():
            if isinstance(v, list): inputs[k] = [x.to(cfg.device) if torch.is_tensor(x) else x for x in v]
            elif torch.is_tensor(v): inputs[k] = v.to(cfg.device)
        try:
            feats, s_ov, s_sal = model(inputs)
            ls = int(inputs["stack_lengths"][0][0])
            stats = desc_loss(inputs["src_pcd_raw"], inputs["tgt_pcd_raw"], feats[:ls], feats[ls:], inputs["correspondences"],
                              inputs["rot"], inputs["trans"], s_ov, s_sal)
            loss = W_CIRCLE * stats["circle_loss"] + W_OVERLAP * stats["overlap_loss"] + W_SALIENCY * stats["saliency_loss"]
            if not torch.isfinite(loss): n_skip += 1; opt.zero_grad(); continue
            loss.backward()
            if not all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None):
                n_skip += 1; opt.zero_grad(); continue
            torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
            opt.step(); opt.zero_grad(); n_ok += 1
            run.append([float(loss), float(stats["circle_loss"]), float(stats["overlap_loss"]), float(stats["saliency_loss"])])
        except RuntimeError as e:
            if "memory" not in str(e).lower(): raise
            opt.zero_grad(); n_skip += 1
            if torch.cuda.is_available(): torch.cuda.empty_cache()
        if it % 100 == 0 and run:
            m = np.mean(run[-100:], 0)
            print(f"epoch {epoch} iter {it}/{len(train_loader)}: loss {m[0]:.3f} (circle {m[1]:.3f}, overlap {m[2]:.3f}, saliency {m[3]:.3f}) "
                  f"| skipped {n_skip} | {(time.time()-t_ep)/60:.1f} min")
        if (time.time() - t_start) / 3600 > TIME_BUDGET_H:
            print("time budget reached: stopping inside epoch", epoch); stop = True; break
    sched.step()
    v_all, v_cross = validate()
    hist.append(dict(epoch=epoch, iters=n_ok, loss=float(np.mean([r_[0] for r_ in run])) if run else np.nan, val_succ=v_all, val_succ_cross=v_cross))
    pd.DataFrame(hist).to_csv(HIST, index=False)
    print(f"== epoch {epoch}: {n_ok} steps, mean loss {hist[-1]['loss']:.3f}, validation success {v_all:.0%} (cross-stage {v_cross:.0%})")
    if v_all > best:
        best = v_all
        torch.save(dict(state_dict=model.state_dict(), scale=SCALE, NL=np.asarray(NL).tolist(), epoch=epoch, val_succ=v_all), BEST)
        print("   new best -> saved", BEST.name)
    torch.save(dict(state_dict=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(), epoch=epoch, best=best), CKPT)
    if stop: break

## 7. Summary

In [ ]:
h = pd.DataFrame(hist); print(h.round(3).to_string())
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].plot(h.epoch, h.loss, "o-", color="#3d85c6"); ax[0].set_xlabel("epoch"); ax[0].set_ylabel("training loss")
ax[1].plot(h.epoch, h.val_succ, "o-", label="all", color="#c96a12"); ax[1].plot(h.epoch, h.val_succ_cross, "s--", label="cross-stage", color="#d62728")
ax[1].set_xlabel("epoch (0 = pretrained)"); ax[1].set_ylabel(f"validation success ({VAL_BUILDING})"); ax[1].set_ylim(0, 1); ax[1].legend()
fig.suptitle("Fig F1. PREDATOR fine-tuning on NSS (train: " + ", ".join(TRAIN_BUILDINGS) + ")")
L.save(fig, FIG / "F1_finetune_curve.png"); plt.show()
best_state = torch.load(BEST, map_location="cpu", weights_only=False)
(OUT / "summary.json").write_text(json.dumps(dict(train_buildings=TRAIN_BUILDINGS, val_building=VAL_BUILDING, scale=SCALE,
    n_train=len(train_rows), n_val=len(val_rows), best_epoch=int(best_state["epoch"]), best_val_succ=float(best_state["val_succ"]),
    pretrained_val_succ=float(h.val_succ.iloc[0]), history=h.round(4).to_dict("records")), indent=1, default=str))
print(f"best epoch {best_state['epoch']} (validation {best_state['val_succ']:.0%}; pretrained {h.val_succ.iloc[0]:.0%}) -> {BEST}")
print("NEXT: add this notebook's output as an input to NB4. NB4 finds predator_nss_ft.pth automatically.")
UPLOAD = L.bundle(WORK, NB, OUT)